# Packages, Functions, and Initial Data Pre-porcession

## Functions

In [ ]:
# # This is our default function, the one we use to prep the data for the encoder that takes us from spectra to ChemNet encodings 
# def create_dataset_tensors(spectra_dataset, embedding_df, device, start_idx=None, stop_idx=None):
#     """
#     Create tensors from the provided spectra dataset and embedding DataFrame.

#     Parameters:
#     ----------
#     spectra_dataset : pd.DataFrame
#         DataFrame containing spectral data and chemical labels. Assumes specific 
#         columns for processing based on the `carl` flag.

#     embedding_df : pd.DataFrame
#         DataFrame containing embeddings for chemicals, with 'Embedding Floats' 
#         column corresponding to ChemNet embeddings.

#     device : torch.device
#         The device (CPU or GPU) on which to store the tensors.

#     carl : bool, optional
#         If True, processes the dataset assuming it has a different structure 
#         (specifically without an 'Unnamed: 0' column). Default is False.

#     Returns:
#     -------
#     tuple
#         A tuple containing:
#         - embeddings_tensor (torch.Tensor): Tensor of true embeddings for the chemicals.
#         - spectra_tensor (torch.Tensor): Tensor of spectral data.
#         - chem_encodings_tensor (torch.Tensor): Tensor of chemical name encodings.
#         - spectra_indices_tensor (torch.Tensor): Tensor of indices corresponding to the spectra.
#     """
#     spectra = spectra_dataset.iloc[:,1:-4] # Exclude 'SMILES_spectra', 'index', and 'Unnamed: 0' columns

#     # create tensors of spectra, true embeddings, and chemical name encodings for train and val
#     chem_labels = list(spectra_dataset['SMILES_spectra'])
#     embeddings_tensor = torch.Tensor([embedding_df.loc[embedding_df['SMILES'] == chem_name].iloc[0, 1:].values.astype(float) for chem_name in chem_labels]).to(device)
#     spectra_tensor = torch.Tensor(spectra.values).to(device)
#     spectra_indices_tensor = torch.Tensor(spectra_dataset['index'].to_numpy()).to(device)

#     return embeddings_tensor, spectra_tensor, spectra_indices_tensor 

def create_dataset_tensors(spectra_dataset, embedding_df, device, start_idx=None, stop_idx=None):
    """
    Create tensors from the provided spectra dataset and embedding DataFrame.
    """
    # Use the provided start_idx and stop_idx for flexible column selection
    if start_idx is None:
        start_idx = 1  # Default: skip SMILES_spectra
    if stop_idx is None:
        stop_idx = -3  # Default: exclude last 3 columns (index_id, Response, log_response)
    
    # Extract spectral features using the provided indices
    spectra = spectra_dataset.iloc[:, start_idx:stop_idx]
    
    # Create tensors
    chem_labels = list(spectra_dataset['SMILES_spectra'])
    embeddings_tensor = torch.Tensor([
        embedding_df.loc[embedding_df['SMILES'] == chem_name].iloc[0, 1:].values.astype(float) 
        for chem_name in chem_labels
    ]).to(device)
    spectra_tensor = torch.Tensor(spectra.values).to(device)
    spectra_indices_tensor = torch.Tensor(spectra_dataset['index'].to_numpy()).to(device)

    return embeddings_tensor, spectra_tensor, spectra_indices_tensor


In [ ]:
def apply_threshold_filter(df, threshold):
    """
    Applies a threshold filter to spectral data, setting values below threshold to zero.
    
    Parameters:
    df: DataFrame with first column as SMILES, last column as index_id, rest as spectral intensity columns
    threshold: Float, minimum value to keep (values below this become 0)
    
    Returns:
    DataFrame with filtered spectral data (values below threshold set to 0)
    """
    
    # Create a copy to avoid modifying the original
    filtered_df = df.copy()
    
    # Get spectral columns (all except first and last column)
    spectral_cols = filtered_df.columns[1:-1]
    
    # Ensure spectral data is numeric
    # filtered_df[spectral_cols] = filtered_df[spectral_cols].apply(pd.to_numeric, errors='coerce')
    
    # Apply threshold using numpy where - more explicit control
    spectral_data = filtered_df[spectral_cols].values
    spectral_data = np.where(spectral_data > threshold, spectral_data, 0)
    filtered_df[spectral_cols] = spectral_data
    
    # index_id column is preserved unchanged
    return filtered_df

In [ ]:
def bin_spectra_by_mz_range(df, bin_size):
    """
    Bins spectra data by grouping m/z columns into ranges of specified size.
    
    Parameters:
    df: DataFrame with first column as SMILES, last column as index_id, rest as m/z columns (float names)
    bin_size: Float, the size of each bin (e.g., 10 means bins of 0-10, 10-20, etc.)
    
    Returns:
    DataFrame with SMILES column, binned m/z columns named by bin midpoints, and index_id column
    """
    smiles_col = df.columns[0]
    index_col = df.columns[-1]  # Preserve the last column (index_id)
    mz_cols = df.columns[1:-1]  # Exclude first and last columns
    
    # Create bins and assign each m/z to a bin
    bin_assignments = {}
    for mz in mz_cols:
        bin_start = (mz // bin_size) * bin_size
        bin_end = bin_start + bin_size
        bin_midpoint = bin_start + (bin_size / 2)
        
        # Round to avoid floating point precision issues
        bin_midpoint = round(bin_midpoint, 3)  
        
        if bin_midpoint not in bin_assignments:
            bin_assignments[bin_midpoint] = []
        bin_assignments[bin_midpoint].append(mz)
    
    # Create new DataFrame with binned data
    result_df = pd.DataFrame()
    result_df[smiles_col] = df[smiles_col]
    
    # Sum intensities for each bin
    for bin_midpoint in sorted(bin_assignments.keys()):
        cols_in_bin = bin_assignments[bin_midpoint]
        result_df[bin_midpoint] = df[cols_in_bin].sum(axis=1)
    
    # Preserve index_id column
    result_df[index_col] = df[index_col]
    
    return result_df

In [ ]:
def fill_missing_bins(df, bin_size):
    """
    Fills in missing bin columns in a binned DataFrame.
    
    Parameters:
    df: DataFrame with first column as SMILES, last column as index_id, rest as binned m/z columns (float names)
    bin_size: Float, the original bin size used for binning
    
    Returns:
    DataFrame with all missing bin midpoints filled in with zeros
    """
    smiles_col = df.columns[0]
    index_col = df.columns[-1]  # Preserve the last column (index_id)
    existing_bins = sorted([col for col in df.columns[1:-1] if isinstance(col, (int, float))])
    
    if not existing_bins:
        return df
    
    # Calculate the step size 
    step_size = bin_size
    
    # Find the range of bins to fill
    min_bin = existing_bins[0]
    max_bin = existing_bins[-1]
    
    # Generate all possible bin midpoints from first non-zero step to max_bin
    all_bins = []
    current_bin = step_size / 2  # Start from first non-zero bin (don't include 0)
    while current_bin <= max_bin:
        all_bins.append(current_bin)
        current_bin += step_size
    
    # Find missing bins
    missing_bins = set(all_bins) - set(existing_bins)
    
    # Add missing bins with zeros
    result_df = df.copy()
    for bin_midpoint in missing_bins:
        result_df[bin_midpoint] = 0.0
    
    # Reorder columns: SMILES column first, then sorted bin columns, then index_id column
    bin_cols = sorted([col for col in result_df.columns[1:-1] if isinstance(col, (int, float))])
    ordered_cols = [smiles_col] + bin_cols + [index_col]
    result_df = result_df[ordered_cols]
    
    return result_df

In [ ]:
# Add the 'Response' and 'log_response' columns 
def add_response_and_log_response(spectra_df, original_df, smiles_col='SMILES_spectra'):
    """
    Adds 'Response' and 'log_response' columns to spectra_df by mapping from original_df using the SMILES column.
    """
    smiles_to_response = original_df.drop_duplicates(subset=smiles_col).set_index(smiles_col)['Response']
    spectra_df['Response'] = spectra_df[smiles_col].map(smiles_to_response)
    spectra_df['log_response'] = np.log(spectra_df['Response'])
    return spectra_df

In [ ]:
# Spectrum string to dataframe function
def spectrum_string_to_dataframe(df, spectrum_col, smiles_col):
    """
    Converts a DataFrame with a spectrum column (string of 'x:y' pairs) into a matrix
    where columns are unique x values, rows are spectra (even for duplicate SMILES), and values are y (intensity).
    The index will match the original DataFrame.
    """
    # Collect all unique x values (m/z)
    x_values_set = set()
    spectra_list = []
    for idx, row in df.iterrows():
        spectrum = row[spectrum_col]
        pairs = spectrum.split()
        xys = []
        for pair in pairs:
            try:
                x, y = pair.split(":") # Split into x and y
                #x = float(x.replace("'", "").replace('"', '')) # Remove quotes and convert to float (done in processing)
                #y = float(y.replace("'", "").replace('"', '')) # Remove quotes and convert to float (done in processing)
                xys.append((x, y))
                x_values_set.add(x)
            except Exception:
                continue
        spectra_list.append((row[smiles_col], dict(xys)))
    x_values = sorted(x_values_set) # Sort the x values to maintain order
    
    # Build the matrix
    matrix = []
    smiles_list = []
    for smiles, xy_dict in spectra_list:
        row = [xy_dict.get(x, 0.0) for x in x_values]
        matrix.append(row)
        smiles_list.append(smiles)
    df_matrix = pd.DataFrame(matrix, columns=[x for x in x_values]) # columns=[f"mz_{x}" for x in x_values]) to make stings
    df_matrix.insert(0, smiles_col, smiles_list)
    df_matrix.index = df.index  # preserve original row order/index
    return df_matrix

## Import

In [ ]:
# Package imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import seaborn as sns

# from fcd_torch import FCD
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA
import os 
from fcd_torch import FCD
import rdkit

import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import functions_enc as f

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

## Initial Pre-processing

In [ ]:
# The 5/30 dataset with rat based toxicity data and groups
df3 = pd.read_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/MIT_LL_data3.csv")
print(df3.shape)
df3.head()

# Uniformity of ionization model labels
print(df3["Ionization_Mode"].unique())
df3["Ionization_Mode"] = df3["Ionization_Mode"].replace("'Positive'", "'positive'")
print(df3["Ionization_Mode"].unique())

# Remove the N/A values in Ionization_Mode
df3 = df3[df3["Ionization_Mode"] != "'NaN'"]
print(df3["Ionization_Mode"].unique())

# Removing the '' from the SMILES
# Remove single quotes from all string columns in df3
df3 = df3.applymap(lambda x: x.replace("'", "") if isinstance(x, str) else x)
#df3["SMILES_spectra"] = df3["SMILES_spectra"].str.replace("'", "")

# This will give us the subsets with all of the relevant information
df3_QQpos = df3[df3['Group'] == 'Q-Orbitrap-positive'] # 1307
# Save df3_QQpos to the MIT_LL_data folder
# df3_QQpos.to_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/df3_QQpos.csv", index=False)
# print(f"Saved df3_QQpos with shape {df3_QQpos.shape} to /home/dlipsey/MITLincolnLabs/MIT_LL_data/df3_QQpos.csv")

# # At the moment we only care about Q-Orbitrap-positive data and not the other instrument and ionization mode groups
# df3_QQneg = df3[df3['Group'] == 'Q-Orbitrap-negative'] # 756
# df3_QTOFpos = df3[df3['Group'] == 'Q-TOF-positive'] # 736  
# df3_LTQOpos = df3[df3['Group'] == 'LTQ-Orbitrap-positive'] # 481 


In [ ]:
df3_QQpos_spectra = spectrum_string_to_dataframe(df3_QQpos, "Spectrum", "SMILES_spectra")

# Data processing using the function
cols = df3_QQpos_spectra.columns.tolist()
# Keep the first column as is, convert the rest to float
new_cols = [cols[0]] + [float(c) for c in cols[1:]]
df3_QQpos_spectra.columns = new_cols
df3_QQpos_spectra.iloc[:, 1:] = df3_QQpos_spectra.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')
all_float = all(isinstance(c, float) for c in df3_QQpos_spectra.columns[1:])
print("All columns are float:", all_float)
# Replace NaN values with 0
df3_QQpos_spectra.iloc[:, 1:] = df3_QQpos_spectra.iloc[:, 1:].fillna(0)


spectra = df3_QQpos_spectra.iloc[:, 1:]

# Check if every element is a float
all_float_elements = spectra.applymap(lambda x: isinstance(x, float)).all().all()
print("All elements are float:", all_float_elements)

# Sort columns by their names 
first_col = df3_QQpos_spectra.columns[0]
sorted_cols = [first_col] + sorted(df3_QQpos_spectra.columns[1:])
df3_QQpos_spectra = df3_QQpos_spectra[sorted_cols]

# Add unique index_id column for outlier tracking
df3_QQpos_spectra['index_id'] = range(len(df3_QQpos_spectra))

print(f"Data processing complete. Shape: {df3_QQpos_spectra.shape}")
print(f"Added index_id column with values from 0 to {len(df3_QQpos_spectra)-1}")
print(f"Columns: SMILES + {len(df3_QQpos_spectra.columns)-2} spectral features + index_id")


In [ ]:
df3_QQpos_spectra.head()

In [ ]:
df3_QQpos_spectra.head()

# Binning Loop

In [ ]:
# # Create ALL binned and thresholded datasets (complete grid search)
# print("Creating all binned and thresholded datasets...")
# df3_QQpos_spectra_original = df3_QQpos_spectra.copy()
# bin_sizes = [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 25, 50, 100, 200, 500, 1000]
# thresholds = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 50, 100]
# import warnings
# with warnings.catch_warnings():
#     warnings.simplefilter("ignore")
        
#     for bin_size in bin_sizes:
#         for threshold in thresholds:
            
#             # Create variable name
#             bin_str = str(bin_size).replace('.', '_')
#             thresh_str = str(threshold).replace('.', '_')
#             var_name = f"bin{bin_str}_thresh{thresh_str}_df3_QQpos_spectra"
                
#             # Start with original data
#             current_data = df3_QQpos_spectra_original.copy()
        
#             # Apply threshold filtering first
#             threshold_filtered_data = apply_threshold_filter(current_data, threshold)
            
#             # Then apply binning
#             binned_data = bin_spectra_by_mz_range(threshold_filtered_data, bin_size)
        
#             # Fill missing bins
#             filled_data = fill_missing_bins(binned_data, bin_size)
        
#             # Add response and log response values
#             final_data = add_response_and_log_response(filled_data, df3_QQpos)
            
#             # Ensure index_id is preserved from original data
#             final_data['index_id'] = df3_QQpos_spectra['index_id'].iloc[:len(final_data)].values
            
#             # Store in globals()
#             globals()[var_name] = final_data
            
#             # Save to file
#             save_path = f"/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/grid_search_dataframes/{var_name}.pkl"
#             final_data.to_pickle(save_path)
#             print(f"Saved {var_name} to {save_path} - Shape: {final_data.shape}")

# print(f"  - {len(bin_sizes)} bin sizes: {bin_sizes}")
# print(f"  - {len(thresholds)} threshold values: {thresholds}")
# print(f"  - Plus the existing {len(bin_sizes)} thresh0 datasets")

In [ ]:
# # Create the missing threhold 0 datasets
# print("Creating binned-only datasets (thresh0)...")
# df3_QQpos_spectra_original = df3_QQpos_spectra.copy()
# bin_sizes = [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 25, 50, 100, 200, 500, 1000]
# with warnings.catch_warnings():
#     warnings.simplefilter("ignore")
    
#     for bin_size in bin_sizes:
#         # Create variable name for thresh0 (no threshold)
#         bin_str = str(bin_size).replace('.', '_')
#         var_name = f"bin{bin_str}_thresh_zero_df3_QQpos_spectra"
    
#         print(f"Creating {var_name}...")
    
#         # Start with original data (no threshold filtering)
#         current_data = df3_QQpos_spectra_original.copy()
    
#         # Binning only
#         binned_data = bin_spectra_by_mz_range(current_data, bin_size)
    
#         # Fill missing bins
#         filled_data = fill_missing_bins(binned_data, bin_size)
    
#         # Add response and log response values
#         final_data = add_response_and_log_response(filled_data, df3_QQpos)
        
#         # Ensure index_id is preserved from original data
#         final_data['index_id'] = df3_QQpos_spectra['index_id'].iloc[:len(final_data)].values
        
#         # Store in globals()
#         globals()[var_name] = final_data
        
#         # Save to file
#         save_path = f"/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/grid_search_dataframes/{var_name}.pkl"
#         final_data.to_pickle(save_path)
#         print(f"Saved {var_name} to {save_path}")

# print(f"Created {len(bin_sizes)} thresh0 datasets!")

# Random Forest on Spectra

## Random Forest Training

In [ ]:
# import os
# import gc
# import pickle

# # Load datasets folder path
# grid_search_folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/grid_search_dataframes"

# # Get all .pkl files in the folder
# pkl_files = [f for f in os.listdir(grid_search_folder) if f.endswith('.pkl')]
# dataset_names = [f.replace('.pkl', '') for f in pkl_files]

# print(f"Found {len(dataset_names)} datasets to process")

# # Initialize storage for results
# results_r2 = []
# results_percent_error = []

# # Process datasets ONE AT A TIME (memory efficient)
# for i, dataset_name in enumerate(sorted(dataset_names), 1):
#     print(f"Processing {i}/{len(dataset_names)}: {dataset_name}")
    
#     try:
#         # Load only the current dataset
#         file_path = os.path.join(grid_search_folder, f"{dataset_name}.pkl")
#         df = pd.read_pickle(file_path)
        
#         # Prepare features and target
#         feature_cols = [col for col in df.columns if col not in ['SMILES_spectra', 'Response', 'log_response', 'index_id']]
#         X = df[feature_cols]
#         y = df['log_response']
        
#         # Remove rows with NaN values
#         valid_mask = ~(X.isna().any(axis=1) | y.isna())
#         X_clean = X[valid_mask]
#         y_clean = y[valid_mask]
        
#         if len(X_clean) < 10:  # Skip if too few samples
#             print(f"  Skipping {dataset_name}: Only {len(X_clean)} valid samples")
#             continue
            
#         # Split the data
#         X_train, X_test, y_train, y_test = train_test_split(
#             X_clean, y_clean, test_size=0.2, random_state=42
#         )
        
#         # Train Random Forest with limited CPU usage
#         rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=2)
#         rf.fit(X_train, y_train)
        
#         # Make predictions
#         y_train_pred = rf.predict(X_train)
#         y_test_pred = rf.predict(X_test)
        
#         # Calculate R² metrics
#         train_r2 = r2_score(y_train, y_train_pred)
#         test_r2 = r2_score(y_test, y_test_pred)
        
#         # Calculate absolute percent error (undo log transform first)
#         y_train_true_response = np.exp(y_train)
#         y_train_pred_response = np.exp(y_train_pred)
#         y_test_true_response = np.exp(y_test)
#         y_test_pred_response = np.exp(y_test_pred)
        
#         # Calculate median and mean absolute percent error
#         train_median_percent_error = 100 * (np.median(np.abs(y_train_pred_response - y_train_true_response) / y_train_true_response))
#         test_median_percent_error = 100 * (np.median(np.abs(y_test_pred_response - y_test_true_response) / y_test_true_response))
#         train_mean_percent_error = 100 * (np.mean(np.abs(y_train_pred_response - y_train_true_response) / y_train_true_response))
#         test_mean_percent_error = 100 * (np.mean(np.abs(y_test_pred_response - y_test_true_response) / y_test_true_response))

#         # Store results
#         results_r2.append({
#             'Dataset': dataset_name,
#             'Train_R2': train_r2,
#             'Test_R2': test_r2,
#             'Samples': len(X_clean),
#             'Features': len(feature_cols)
#         })
        
#         results_percent_error.append({
#             'Dataset': dataset_name,
#             'Train_Median_Percent_Error': train_median_percent_error,
#             'Test_Median_Percent_Error': test_median_percent_error,
#             'Train_Mean_Percent_Error': train_mean_percent_error,
#             'Test_Mean_Percent_Error': test_mean_percent_error,
#             'Samples': len(X_clean),
#             'Features': len(feature_cols)
#         })
        
#         print(f"Completed: Test R² = {test_r2:.4f}, Test Median % Error = {test_median_percent_error:.1f}%")
        
#     except Exception as e:
#         print(f"Error processing {dataset_name}: {str(e)}")
#         continue
    
#     finally:
#         # Always clean up memory after each dataset
#         if 'df' in locals():
#             del df
#         if 'X' in locals():
#             del X, y, X_clean, y_clean
#         if 'rf' in locals():
#             del rf
#         gc.collect()
        
#         # Periodic deeper cleanup every 20 datasets
#         if i % 20 == 0:
#             print(f"  Deep cleanup after {i} datasets...")
#             gc.collect()

# # Verify we have the right count
# thresh0_datasets = [name for name in dataset_names if 'thresh_zero' in name]
# thresholded_datasets = [name for name in dataset_names if 'thresh_zero' not in name]

# print(f"  - Datasets with thresh0 (no threshold): {len(thresh0_datasets)}")
# print(f"  - Datasets with thresholds applied: {len(thresholded_datasets)}")

# # Convert results to DataFrames
# df_r2_results = pd.DataFrame(results_r2)
# df_percent_error_results = pd.DataFrame(results_percent_error)

# print(f"\nCompleted! Processed {len(results_r2)} datasets successfully.")
# print(f"Results stored in: df_r2_results, df_percent_error_results")

In [ ]:
import os
import gc
import pickle

# Load datasets folder path
grid_search_folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/grid_search_dataframes"

# Get all .pkl files in the folder
pkl_files = [f for f in os.listdir(grid_search_folder) if f.endswith('.pkl')]
dataset_names = [f.replace('.pkl', '') for f in pkl_files]

print(f"Found {len(dataset_names)} datasets to process")
print(f"Expected: 182 datasets")

# Verify we have the right count
thresh0_datasets = [name for name in dataset_names if 'thresh_zero' in name]
thresholded_datasets = [name for name in dataset_names if 'thresh_zero' not in name]

print(f"  - Datasets with thresh0 (no threshold): {len(thresh0_datasets)}")
print(f"  - Datasets with thresholds applied: {len(thresholded_datasets)}")

# Initialize storage for results
results_r2 = []
results_percent_error = []

# Dictionary to store individual errors for histogram analysis
saved_spectral_errors = {}

# Process datasets ONE AT A TIME (memory efficient)
for i, dataset_name in enumerate(sorted(dataset_names), 1):
    print(f"Processing {i}/{len(dataset_names)}: {dataset_name}")
    
    try:
        # Load only the current dataset
        file_path = os.path.join(grid_search_folder, f"{dataset_name}.pkl")
        df = pd.read_pickle(file_path)
        
        # Prepare features and target
        feature_cols = [col for col in df.columns if col not in ['SMILES_spectra', 'Response', 'log_response', 'index_id']]
        X = df[feature_cols]
        y = df['log_response']
        
        # Remove rows with NaN values
        valid_mask = ~(X.isna().any(axis=1) | y.isna())
        X_clean = X[valid_mask]
        y_clean = y[valid_mask]
        
        if len(X_clean) < 10:  # Skip if too few samples
            print(f"  Skipping {dataset_name}: Only {len(X_clean)} valid samples")
            continue
            
        # Split the data
        X_train, X_test, y_train, y_test = train_test_split(
            X_clean, y_clean, test_size=0.2, random_state=42
        )
        
        # Train Random Forest with limited CPU usage
        rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=2)
        rf.fit(X_train, y_train)
        
        # Make predictions
        y_train_pred = rf.predict(X_train)
        y_test_pred = rf.predict(X_test)
        
        # Calculate R² metrics
        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)
        
        # Calculate absolute percent error (undo log transform first)
        y_train_true_response = np.exp(y_train)
        y_train_pred_response = np.exp(y_train_pred)
        y_test_true_response = np.exp(y_test)
        y_test_pred_response = np.exp(y_test_pred)
        
        # Calculate individual errors for test set
        individual_errors = np.abs((y_test_pred_response - y_test_true_response) / y_test_true_response) * 100
        
        # Save individual errors for histogram analysis
        saved_spectral_errors[dataset_name] = individual_errors
        
        # Calculate median and mean absolute percent error
        train_median_percent_error = 100 * (np.median(np.abs(y_train_pred_response - y_train_true_response) / y_train_true_response))
        test_median_percent_error = 100 * (np.median(np.abs(y_test_pred_response - y_test_true_response) / y_test_true_response))
        train_mean_percent_error = 100 * (np.mean(np.abs(y_train_pred_response - y_train_true_response) / y_train_true_response))
        test_mean_percent_error = 100 * (np.mean(np.abs(y_test_pred_response - y_test_true_response) / y_test_true_response))

        # Store results
        results_r2.append({
            'Dataset': dataset_name,
            'Train_R2': train_r2,
            'Test_R2': test_r2,
            'Samples': len(X_clean),
            'Features': len(feature_cols)
        })
        
        results_percent_error.append({
            'Dataset': dataset_name,
            'Train_Median_Percent_Error': train_median_percent_error,
            'Test_Median_Percent_Error': test_median_percent_error,
            'Train_Mean_Percent_Error': train_mean_percent_error,
            'Test_Mean_Percent_Error': test_mean_percent_error,
            'Samples': len(X_clean),
            'Features': len(feature_cols)
        })
        
        print(f"Completed: Test R² = {test_r2:.4f}, Test Median % Error = {test_median_percent_error:.1f}%")
        
    except Exception as e:
        print(f"Error processing {dataset_name}: {str(e)}")
        continue
    
    finally:
        # Always clean up memory after each dataset
        if 'df' in locals():
            del df
        if 'X' in locals():
            del X, y, X_clean, y_clean
        if 'rf' in locals():
            del rf
        gc.collect()
        
        # Periodic deeper cleanup every 20 datasets
        if i % 20 == 0:
            print(f"  Deep cleanup after {i} datasets...")
            gc.collect()

# Convert results to DataFrames
df_r2_results = pd.DataFrame(results_r2)
df_percent_error_results = pd.DataFrame(results_percent_error)

print(f"\nCompleted! Processed {len(results_r2)} datasets successfully.")
print(f"Saved individual errors for {len(saved_spectral_errors)} datasets")
print(f"Results stored in: df_r2_results, df_percent_error_results")

# Display summary statistics
print("\n=== SUMMARY STATISTICS ===")
print("Test R² Statistics:")
print(df_r2_results['Test_R2'].describe())

print("\nTest Median Percent Error Statistics:")
print(df_percent_error_results['Test_Median_Percent_Error'].describe())

print("\nTest Mean Percent Error Statistics:")
print(df_percent_error_results['Test_Mean_Percent_Error'].describe())

# Show top 10 performing datasets by Test R²
print("\n=== TOP 10 DATASETS BY TEST R² ===")
top_r2 = df_r2_results.nlargest(10, 'Test_R2')[['Dataset', 'Test_R2', 'Features']]
print(top_r2.to_string(index=False))

# Show comparison between thresh0 (no threshold) and thresholded datasets
print("\n=== THRESH0 vs THRESHOLDED COMPARISON ===")
thresh0_results = df_r2_results[df_r2_results['Dataset'].str.contains('thresh_zero')]
thresholded_results = df_r2_results[~df_r2_results['Dataset'].str.contains('thresh_zero')]

if len(thresh0_results) > 0:
    print(f"Thresh0 datasets (no threshold) - Mean Test R²: {thresh0_results['Test_R2'].mean():.4f}")
    print(f"Best thresh0 dataset: {thresh0_results.loc[thresh0_results['Test_R2'].idxmax(), 'Dataset']} (R² = {thresh0_results['Test_R2'].max():.4f})")

if len(thresholded_results) > 0:
    print(f"Thresholded datasets - Mean Test R²: {thresholded_results['Test_R2'].mean():.4f}")
    print(f"Best thresholded dataset: {thresholded_results.loc[thresholded_results['Test_R2'].idxmax(), 'Dataset']} (R² = {thresholded_results['Test_R2'].max():.4f})")

## Spectra Heatmap

In [ ]:
# Create the actual heatmaps for visualization
# First, let's extract bin size and threshold from the dataset names and add to results
def parse_dataset_name(dataset_name):
    """Extract bin size and threshold from dataset name"""
    # Handle thresh_zero case (no threshold)
    if 'thresh_zero' in dataset_name:
        # Extract bin size
        bin_part = dataset_name.split('_thresh_zero')[0].replace('bin', '')
        bin_size = float(bin_part.replace('_', '.'))
        threshold = 0.0
    else:
        # Extract bin size and threshold
        parts = dataset_name.split('_thresh')
        bin_part = parts[0].replace('bin', '')
        bin_size = float(bin_part.replace('_', '.'))
        
        thresh_part = parts[1].split('_df3_QQpos_spectra')[0]
        threshold = float(thresh_part.replace('_', '.'))
    
    return bin_size, threshold

# Add bin_size and threshold columns to results DataFrames
for df_results in [df_r2_results, df_percent_error_results]:
    bin_sizes = []
    thresholds = []
    
    for dataset_name in df_results['Dataset']:
        bin_size, threshold = parse_dataset_name(dataset_name)
        bin_sizes.append(bin_size)
        thresholds.append(threshold)
    
    df_results['BinSize'] = bin_sizes
    df_results['Threshold'] = thresholds

# Check for and remove duplicates before creating pivot tables
print("Checking for duplicates in results...")
print(f"Original df_r2_results shape: {df_r2_results.shape}")

# Remove duplicates based on BinSize + Threshold combination (keep first occurrence)
df_r2_results = df_r2_results.drop_duplicates(subset=['BinSize', 'Threshold'], keep='first')
df_percent_error_results = df_percent_error_results.drop_duplicates(subset=['BinSize', 'Threshold'], keep='first')

print(f"After removing duplicates: {df_r2_results.shape}")

# Now create pivot tables 
r2_pivot = df_r2_results.pivot(index='BinSize', columns='Threshold', values='Test_R2') 
median_percent_error_pivot = df_percent_error_results.pivot(index='BinSize', columns='Threshold', values='Test_Median_Percent_Error')
mean_percent_error_pivot = df_percent_error_results.pivot(index='BinSize', columns='Threshold', values='Test_Mean_Percent_Error')

# List all expected thresholds
thresholds_subset = [0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 50, 100]
bins_subset = [0.05, 0.1, 0.5, 1, 2, 5, 10, 25, 50, 100, 200, 500, 1000]

# Reindex pivot tables to show all columns, filling missing with NaN
r2_pivot = r2_pivot.reindex(columns=thresholds_subset, index=bins_subset)
median_percent_error_pivot = median_percent_error_pivot.reindex(columns=thresholds_subset, index=bins_subset)
mean_percent_error_pivot = mean_percent_error_pivot.reindex(columns=thresholds_subset, index=bins_subset)

# Also create individual larger heatmaps for better detail
def create_detailed_heatmap_spec(pivot_data, metric_name, cmap, figsize=(12, 8), vmin=None, vmax=None):
    """Create a detailed heatmap for a single metric"""
    plt.figure(figsize=figsize)
    
    # Create heatmap
    sns.heatmap(pivot_data, 
                annot=True, 
                fmt='.3f' if 'R²' in metric_name else '.1f', 
                cmap=cmap,
                square=False,
                linewidths=0.5,
                vmin=vmin,
                vmax=vmax,
                cbar_kws={'label': f'Test {metric_name}', 'shrink': 0.8})
    
    plt.title(f'Spectra: {metric_name} by Bin Size and Threshold', fontsize=16, fontweight='bold')
    plt.xlabel('Threshold Value', fontsize=14)
    plt.ylabel('Bin Size', fontsize=14)
    plt.gca().invert_yaxis()
    
    # Improve readability
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    
    # Add text annotation for best performance
    if 'R²' in metric_name:
        best_val = pivot_data.max().max()
        plt.text(0.02, 0.98, f'Best R²: {best_val:.4f}', 
                transform=plt.gca().transAxes, 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                verticalalignment='top')
    else:
        best_val = pivot_data.min().min()
        plt.text(0.02, 0.98, f'Best {metric_name}: {best_val:.1f}%', 
                transform=plt.gca().transAxes, 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                verticalalignment='top')
    
    plt.tight_layout()
    plt.savefig(f"/home/dlipsey/MITLincolnLabs/Figures/Spectra_{metric_name}_by_Bin_Size_and_Threshold")
    plt.show()

# Create detailed individual heatmaps
print("Creating detailed heatmaps...")

# create_detailed_heatmap_spec(r2_pivot, 'R²', 'RdYlBu')     
create_detailed_heatmap_spec(median_percent_error_pivot, 'Median_Absolute_Percent_Error', 'RdYlBu_r', vmin=1.0, vmax=100.0) 
create_detailed_heatmap_spec(mean_percent_error_pivot, 'Mean_Absolute_Percent_Error', 'RdYlBu_r', vmin=1.0, vmax=100.0)

# Encoder

## Architecture

In [ ]:
batch_size = 64
epochs=500
lr=0.0001
criterion=nn.MSELoss()
output_size = 512
num_layers = 5

#%%
# Encoder architecture (With Validation Set)
class Encoder(nn.Module):
    def __init__(self, input_size, output_size, num_layers):
        super().__init__()
        layers = []
        layer_sizes = np.linspace(input_size, output_size, num_layers + 1, dtype=int)
        for i in range(num_layers):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < num_layers - 1:
                layers.append(nn.LeakyReLU(inplace=True))
        self.encoder = nn.Sequential(*layers)

    def forward(self, x):
        return self.encoder(x)

def train_model_encoder(model, train_data, val_data, epochs, learning_rate, criterion, device):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch, true_embeddings, _ in train_data:
            batch = batch.to(device)
            true_embeddings = true_embeddings.to(device)

            optimizer.zero_grad()
            batch_predicted_embeddings = model(batch)
            loss = criterion(batch_predicted_embeddings, true_embeddings) # loss1 (embedding loss) and loss2 (toxicity loss)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        average_train_loss = running_loss / len(train_loader_enc)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for val_batch, val_true_embeddings, _ in val_data:
                val_batch = val_batch.to(device)
                val_true_embeddings = val_true_embeddings.to(device)

                val_batch_predicted_embeddings = model(val_batch)

                val_loss = criterion(val_batch_predicted_embeddings, val_true_embeddings)
                val_loss += loss.item()
        average_val_loss = val_loss / len(val_loader_enc)

        print(f'Epoch [{epoch+1}/{epochs}]')
        print(f'   Training loss: {average_train_loss}')
        print(f'   Validation loss: {average_val_loss}')

    return model
#%%


## Encoder Training

In [ ]:
# Create folder for ChemNet datasets
chemnet_folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/chemnet_grid_search_dataframes"
#os.makedirs(chemnet_folder, exist_ok=True)
#print(f"Created folder: {chemnet_folder}")

In [ ]:
# # Test the encoder loop on just one dataset
# device = f.set_up_gpu()
# name_smiles_embedding_df = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QQpos_no_repeats.csv")

# # Load just one dataset for testing
# grid_search_folder = "/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/grid_search_dataframes"
# test_dataset_name = "bin0_01_thresh0_001_df3_QQpos_spectra"
# test_file_path = os.path.join(grid_search_folder, f"{test_dataset_name}.pkl")
# current_dataset = pd.read_pickle(test_file_path)

# print(f"Testing with dataset: {test_dataset_name}")
# print(f"Original dataset shape: {current_dataset.shape}")
# print(f"Original columns: {current_dataset.columns.tolist()}")

# # Fix data types
# for col in current_dataset.columns:
#     if col not in ['SMILES_spectra', 'index_id']:
#         current_dataset[col] = pd.to_numeric(current_dataset[col], errors='coerce').fillna(0.0)
#         current_dataset[col] = current_dataset[col].astype(np.float32)

# # Apply train/test split
# counts = current_dataset['SMILES_spectra'].value_counts()
# valid_smiles = counts[counts >= 4].index
# filtered_dataset = current_dataset[current_dataset['SMILES_spectra'].isin(valid_smiles)].copy()

# print(f"After filtering (>=4 spectra per SMILES): {filtered_dataset.shape}")

# train_indices = []
# test_indices = []

# np.random.seed(42)
# for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
#     idx = group.index.tolist()
#     n = len(idx)
#     np.random.shuffle(idx)
#     split = n // 2
#     test_idx = idx[:split]
#     train_idx = idx[split:]
#     train_indices.extend(train_idx)
#     test_indices.extend(test_idx)

# train_data_current = filtered_dataset.loc[train_indices].reset_index(drop=True)
# test_data_current = filtered_dataset.loc[test_indices].reset_index(drop=True)
# train_data_current['index'] = train_data_current.index
# test_data_current['index'] = test_data_current.index

# print(f"Train data shape: {train_data_current.shape}")
# print(f"Test data shape: {test_data_current.shape}")

# # Create tensors
# y_train_enc, x_train_enc, train_indices_tensor = create_dataset_tensors(
#     train_data_current, name_smiles_embedding_df, device, start_idx=1, stop_idx=-3)

# y_val_enc, x_val_enc, val_indices_tensor = create_dataset_tensors(
#     test_data_current, name_smiles_embedding_df, device, start_idx=1, stop_idx=-3)

# print(f"Training tensor shapes:")
# print(f"  x_train_enc: {x_train_enc.shape}")
# print(f"  y_train_enc: {y_train_enc.shape}")
# print(f"  train_indices_tensor: {train_indices_tensor.shape}")

# # Quick encoder training (just a few epochs for testing)
# batch_size = 64
# output_size = 512
# num_layers = 5

# train_dataset = TensorDataset(x_train_enc, y_train_enc, train_indices_tensor)
# val_dataset = TensorDataset(x_val_enc, y_val_enc, val_indices_tensor)
# train_loader_enc = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# val_loader_enc = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# # Create and train encoder (just 5 epochs for testing)
# encoder_current = Encoder(input_size=x_train_enc.shape[1], output_size=output_size, num_layers=num_layers).to(device)

# # Train for just a few epochs to test
# optimizer = torch.optim.Adam(encoder_current.parameters(), lr=0.0001)
# criterion = nn.MSELoss()

# print("Training encoder for 5 epochs (test)...")
# for epoch in range(5):
#     encoder_current.train()
#     running_loss = 0.0
#     for batch_x, batch_y, _ in train_loader_enc:
#         batch_x = batch_x.to(device)
#         batch_y = batch_y.to(device)
        
#         optimizer.zero_grad()
#         predicted = encoder_current(batch_x)
#         loss = criterion(predicted, batch_y)
#         loss.backward()
#         optimizer.step()
#         running_loss += loss.item()
    
#     avg_loss = running_loss / len(train_loader_enc)
#     print(f"Epoch {epoch+1}/5, Loss: {avg_loss:.6f}")

# # Generate embeddings
# encoder_current.eval()
# with torch.no_grad():
#     train_embeddings = encoder_current(x_train_enc).cpu().numpy()
#     test_embeddings = encoder_current(x_val_enc).cpu().numpy()

# print(f"Generated embeddings shapes:")
# print(f"  train_embeddings: {train_embeddings.shape}")
# print(f"  test_embeddings: {test_embeddings.shape}")

# # Create ChemNet dataset
# train_chemnet_df = pd.DataFrame(train_embeddings, columns=[f'emb_{j}' for j in range(output_size)])
# train_chemnet_df['SMILES_spectra'] = train_data_current['SMILES_spectra'].values
# train_chemnet_df['Response'] = train_data_current['Response'].values
# train_chemnet_df['log_response'] = train_data_current['log_response'].values
# train_chemnet_df['index_id'] = train_data_current['index_id'].values

# test_chemnet_df = pd.DataFrame(test_embeddings, columns=[f'emb_{j}' for j in range(output_size)])
# test_chemnet_df['SMILES_spectra'] = test_data_current['SMILES_spectra'].values
# test_chemnet_df['Response'] = test_data_current['Response'].values
# test_chemnet_df['log_response'] = test_data_current['log_response'].values
# test_chemnet_df['index_id'] = test_data_current['index_id'].values

# # Combine train and test
# full_chemnet_df = pd.concat([train_chemnet_df, test_chemnet_df], ignore_index=True)

# print(f"\n=== FINAL CHEMNET DATASET ===")
# print(f"Shape: {full_chemnet_df.shape}")
# print(f"Columns: {full_chemnet_df.columns.tolist()}")
# print("\nFirst few rows:")
# print(full_chemnet_df.head())

# # Save to new folder
# chemnet_dataset_name = f"chemnet_emb_{test_dataset_name}"
# save_path = os.path.join(chemnet_folder, f"{chemnet_dataset_name}.pkl")
# full_chemnet_df.to_pickle(save_path)
# print(f"\nSaved to: {save_path}")

# print("\nTest completed successfully!")

In [ ]:
# # Create folder for ChemNet datasets
# import os
# chemnet_folder = "/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/chemnet_grid_search_dataframes"
# # os.makedirs(chemnet_folder, exist_ok=True)
# # print(f"Created folder: {chemnet_folder}")

# # ENCODER TRAINING LOOP - Process all datasets
# device = f.set_up_gpu()
# name_smiles_embedding_df = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QQpos_no_repeats.csv")

# # Get all dataset files from the grid search folder
# grid_search_folder = "/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/grid_search_dataframes"
# dataset_files = [f for f in os.listdir(grid_search_folder) if f.endswith('.pkl') and 'df3_QQpos_spectra' in f]
# dataset_names = [f.replace('.pkl', '') for f in dataset_files]

# print(f"Found {len(dataset_names)} datasets to process")

# # Storage for encoder results
# encoder_results = []
# chemnet_datasets = {}

# # Training parameters
# batch_size = 64
# output_size = 512
# num_layers = 5

# # Loop through each dataset
# for i, dataset_name in enumerate(sorted(dataset_names), 1):
#     print(f"\nProcessing {i}/{len(dataset_names)}: {dataset_name}")
    
#     try:
#         # Load dataset from pickle file
#         dataset_path = os.path.join(grid_search_folder, f"{dataset_name}.pkl")
#         current_dataset = pd.read_pickle(dataset_path)
        
#         print(f"Loaded {dataset_name} - Shape: {current_dataset.shape}")
        
#         # Fix data types
#         for col in current_dataset.columns:
#             if col not in ['SMILES_spectra', 'index_id']:
#                 current_dataset[col] = pd.to_numeric(current_dataset[col], errors='coerce').fillna(0.0)
#                 current_dataset[col] = current_dataset[col].astype(np.float32)
        
#         # Apply train/test split
#         counts = current_dataset['SMILES_spectra'].value_counts()
#         valid_smiles = counts[counts >= 4].index
#         filtered_dataset = current_dataset[current_dataset['SMILES_spectra'].isin(valid_smiles)].copy()
        
#         print(f"After filtering (>=4 spectra per SMILES): {filtered_dataset.shape}")
        
#         train_indices = []
#         test_indices = []
        
#         np.random.seed(42)
#         for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
#             idx = group.index.tolist()
#             n = len(idx)
#             np.random.shuffle(idx)
#             split = n // 2
#             test_idx = idx[:split]
#             train_idx = idx[split:]
#             train_indices.extend(train_idx)
#             test_indices.extend(test_idx)
        
#         train_data_current = filtered_dataset.loc[train_indices].reset_index(drop=True)
#         test_data_current = filtered_dataset.loc[test_indices].reset_index(drop=True)
#         train_data_current['index'] = train_data_current.index
#         test_data_current['index'] = test_data_current.index
        
#         print(f"Train data shape: {train_data_current.shape}")
#         print(f"Test data shape: {test_data_current.shape}")
        
#         # Create tensors
#         y_train_enc, x_train_enc, train_indices_tensor = create_dataset_tensors(
#             train_data_current, name_smiles_embedding_df, device, start_idx=1, stop_idx=-3)
        
#         y_val_enc, x_val_enc, val_indices_tensor = create_dataset_tensors(
#             test_data_current, name_smiles_embedding_df, device, start_idx=1, stop_idx=-3)
        
#         print(f"Training tensor shapes: x_train: {x_train_enc.shape}, y_train: {y_train_enc.shape}")
        
#         train_dataset = TensorDataset(x_train_enc, y_train_enc, train_indices_tensor)
#         val_dataset = TensorDataset(x_val_enc, y_val_enc, val_indices_tensor)
#         train_loader_enc = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#         val_loader_enc = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
#         # Create and train encoder
#         encoder_current = Encoder(input_size=x_train_enc.shape[1], output_size=output_size, num_layers=num_layers).to(device)
        
#         # Train encoder (using your existing train_model_encoder function)
#         trained_encoder = train_model_encoder(
#             model=encoder_current,
#             train_data=train_loader_enc,
#             val_data=val_loader_enc,
#             epochs=epochs,
#             learning_rate=lr,
#             criterion=criterion,
#             device=device
#         )
        
#         # Generate embeddings
#         encoder_current.eval()
#         with torch.no_grad():
#             train_embeddings = encoder_current(x_train_enc).cpu().numpy()
#             test_embeddings = encoder_current(x_val_enc).cpu().numpy()
        
#         print(f"Generated embeddings shapes: train: {train_embeddings.shape}, test: {test_embeddings.shape}")
        
#         # Create ChemNet dataset with embeddings
#         train_chemnet_df = pd.DataFrame(train_embeddings, columns=[f'emb_{j}' for j in range(output_size)])
#         train_chemnet_df['SMILES_spectra'] = train_data_current['SMILES_spectra'].values
#         train_chemnet_df['Response'] = train_data_current['Response'].values
#         train_chemnet_df['log_response'] = train_data_current['log_response'].values
#         train_chemnet_df['index_id'] = train_data_current['index_id'].values
        
#         test_chemnet_df = pd.DataFrame(test_embeddings, columns=[f'emb_{j}' for j in range(output_size)])
#         test_chemnet_df['SMILES_spectra'] = test_data_current['SMILES_spectra'].values
#         test_chemnet_df['Response'] = test_data_current['Response'].values
#         test_chemnet_df['log_response'] = test_data_current['log_response'].values
#         test_chemnet_df['index_id'] = test_data_current['index_id'].values
        
#         # Combine train and test
#         full_chemnet_df = pd.concat([train_chemnet_df, test_chemnet_df], ignore_index=True)
        
#         print(f"Final ChemNet dataset shape: {full_chemnet_df.shape}")
#         print(f"Columns: {full_chemnet_df.columns.tolist()}")
        
#         # Save to chemnet folder
#         chemnet_dataset_name = f"chemnet_emb_{dataset_name}"
#         save_path = os.path.join(chemnet_folder, f"{chemnet_dataset_name}.pkl")
#         full_chemnet_df.to_pickle(save_path)
#         print(f"Saved to: {save_path}")
        
#         # Store results
#         encoder_results.append({
#             'Original_Dataset': dataset_name,
#             'ChemNet_Dataset': chemnet_dataset_name,
#             'Train_Samples': len(train_data_current),
#             'Test_Samples': len(test_data_current),
#             'Total_Samples': len(full_chemnet_df),
#             'Embedding_Dim': output_size,
#             'Original_Features': x_train_enc.shape[1]
#         })
        
#         # Store in memory as well
#         chemnet_datasets[chemnet_dataset_name] = full_chemnet_df
        
#     except Exception as e:
#         print(f"Error processing {dataset_name}: {str(e)}")
#         continue

# print(f"\n=== ENCODER TRAINING COMPLETED ===")
# print(f"Successfully processed {len(encoder_results)} datasets")
# print(f"Created ChemNet embedding datasets saved to: {chemnet_folder}")

# # Display results summary
# results_df = pd.DataFrame(encoder_results)
# print("\nProcessing Summary:")
# print(results_df)

In [ ]:
# # RETRAIN CHEMNET EMBEDDINGS FOR THRESHOLD 50 AND 100 DATASETS
# print("=== RETRAINING CHEMNET EMBEDDINGS FOR HIGH THRESHOLDS ===")

# # Set up device and load ChemNet reference
# device = f.set_up_gpu()
# name_smiles_embedding_df = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QQpos_no_repeats.csv")

# # Define folders
# grid_search_folder = "/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/grid_search_dataframes"
# chemnet_folder = "/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/chemnet_grid_search_dataframes"

# # Find all datasets with threshold 50 or 100
# target_thresholds = [50, 100]
# pkl_files = [f for f in os.listdir(grid_search_folder) if f.endswith('.pkl')]
# dataset_names = [f.replace('.pkl', '') for f in pkl_files]

# # Filter for threshold 50 and 100 datasets
# target_datasets = []
# for dataset_name in dataset_names:
#     for threshold in target_thresholds:
#         if f'thresh{threshold}_df3_QQpos_spectra' in dataset_name:
#             target_datasets.append(dataset_name)

# print(f"Found {len(target_datasets)} datasets with thresholds 50 or 100 to retrain:")
# for dataset in sorted(target_datasets):
#     print(f"  {dataset}")

# # Training parameters
# batch_size = 64
# output_size = 512
# num_layers = 5

# # Storage for retraining results
# retrain_results = []

# # Loop through target datasets and retrain ChemNet embeddings
# for i, dataset_name in enumerate(sorted(target_datasets), 1):
#     print(f"\nRetraining {i}/{len(target_datasets)}: {dataset_name}")
    
#     try:
#         # Load dataset from pickle file
#         dataset_path = os.path.join(grid_search_folder, f"{dataset_name}.pkl")
#         current_dataset = pd.read_pickle(dataset_path)
        
#         print(f"Loaded {dataset_name} - Shape: {current_dataset.shape}")
        
#         # Fix data types
#         for col in current_dataset.columns:
#             if col not in ['SMILES_spectra', 'index_id']:
#                 current_dataset[col] = pd.to_numeric(current_dataset[col], errors='coerce').fillna(0.0)
#                 current_dataset[col] = current_dataset[col].astype(np.float32)
        
#         # Apply train/test split
#         counts = current_dataset['SMILES_spectra'].value_counts()
#         valid_smiles = counts[counts >= 4].index
#         filtered_dataset = current_dataset[current_dataset['SMILES_spectra'].isin(valid_smiles)].copy()
        
#         print(f"After filtering (>=4 spectra per SMILES): {filtered_dataset.shape}")
        
#         if len(filtered_dataset) < 20:  # Skip if too few samples
#             print(f"Skipping {dataset_name}: Only {len(filtered_dataset)} samples after filtering")
#             continue
        
#         train_indices = []
#         test_indices = []
        
#         np.random.seed(42)
#         for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
#             idx = group.index.tolist()
#             n = len(idx)
#             np.random.shuffle(idx)
#             split = n // 2
#             test_idx = idx[:split]
#             train_idx = idx[split:]
#             train_indices.extend(train_idx)
#             test_indices.extend(test_idx)
        
#         train_data_current = filtered_dataset.loc[train_indices].reset_index(drop=True)
#         test_data_current = filtered_dataset.loc[test_indices].reset_index(drop=True)
#         train_data_current['index'] = train_data_current.index
#         test_data_current['index'] = test_data_current.index
        
#         print(f"Train data shape: {train_data_current.shape}")
#         print(f"Test data shape: {test_data_current.shape}")
        
#         # Create tensors
#         y_train_enc, x_train_enc, train_indices_tensor = create_dataset_tensors(
#             train_data_current, name_smiles_embedding_df, device, start_idx=1, stop_idx=-3)
        
#         y_val_enc, x_val_enc, val_indices_tensor = create_dataset_tensors(
#             test_data_current, name_smiles_embedding_df, device, start_idx=1, stop_idx=-3)
        
#         print(f"Training tensor shapes: x_train: {x_train_enc.shape}, y_train: {y_train_enc.shape}")
        
#         train_dataset = TensorDataset(x_train_enc, y_train_enc, train_indices_tensor)
#         val_dataset = TensorDataset(x_val_enc, y_val_enc, val_indices_tensor)
#         train_loader_enc = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#         val_loader_enc = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
#         # Create and train encoder
#         encoder_current = Encoder(input_size=x_train_enc.shape[1], output_size=output_size, num_layers=num_layers).to(device)
        
#         # Train encoder (using your existing train_model_encoder function)
#         trained_encoder = train_model_encoder(
#             model=encoder_current,
#             train_data=train_loader_enc,
#             val_data=val_loader_enc,
#             epochs=epochs,
#             learning_rate=lr,
#             criterion=criterion,
#             device=device
#         )
        
#         # Generate embeddings
#         encoder_current.eval()
#         with torch.no_grad():
#             train_embeddings = encoder_current(x_train_enc).cpu().numpy()
#             test_embeddings = encoder_current(x_val_enc).cpu().numpy()
        
#         print(f"Generated embeddings shapes: train: {train_embeddings.shape}, test: {test_embeddings.shape}")
        
#         # Create ChemNet dataset with embeddings
#         train_chemnet_df = pd.DataFrame(train_embeddings, columns=[f'emb_{j}' for j in range(output_size)])
#         train_chemnet_df['SMILES_spectra'] = train_data_current['SMILES_spectra'].values
#         train_chemnet_df['Response'] = train_data_current['Response'].values
#         train_chemnet_df['log_response'] = train_data_current['log_response'].values
#         train_chemnet_df['index_id'] = train_data_current['index_id'].values
        
#         test_chemnet_df = pd.DataFrame(test_embeddings, columns=[f'emb_{j}' for j in range(output_size)])
#         test_chemnet_df['SMILES_spectra'] = test_data_current['SMILES_spectra'].values
#         test_chemnet_df['Response'] = test_data_current['Response'].values
#         test_chemnet_df['log_response'] = test_data_current['log_response'].values
#         test_chemnet_df['index_id'] = test_data_current['index_id'].values
        
#         # Combine train and test
#         full_chemnet_df = pd.concat([train_chemnet_df, test_chemnet_df], ignore_index=True)
        
#         print(f"Final ChemNet dataset shape: {full_chemnet_df.shape}")
        
#         # Save to chemnet folder (REPLACE existing file)
#         chemnet_dataset_name = f"chemnet_emb_{dataset_name}"
#         save_path = os.path.join(chemnet_folder, f"{chemnet_dataset_name}.pkl")
#         full_chemnet_df.to_pickle(save_path)
#         print(f"REPLACED: {save_path}")
        
#         # Update global variable if it exists
#         if chemnet_dataset_name in globals():
#             globals()[chemnet_dataset_name] = full_chemnet_df
#             print(f"Updated global variable: {chemnet_dataset_name}")
        
#         # Store results
#         retrain_results.append({
#             'Original_Dataset': dataset_name,
#             'ChemNet_Dataset': chemnet_dataset_name,
#             'Train_Samples': len(train_data_current),
#             'Test_Samples': len(test_data_current),
#             'Total_Samples': len(full_chemnet_df),
#             'Embedding_Dim': output_size,
#             'Original_Features': x_train_enc.shape[1],
#             'Status': 'RETRAINED'
#         })
        
#         print(f"Successfully retrained {chemnet_dataset_name}")
        
#     except Exception as e:
#         print(f"Error retraining {dataset_name}: {str(e)}")
#         retrain_results.append({
#             'Original_Dataset': dataset_name,
#             'ChemNet_Dataset': f"chemnet_emb_{dataset_name}",
#             'Status': f'ERROR: {str(e)}'
#         })
#         continue

# print(f"\n=== RETRAINING COMPLETED ===")
# print(f"Successfully retrained {len([r for r in retrain_results if r.get('Status') == 'RETRAINED'])} datasets")
# print(f"Errors: {len([r for r in retrain_results if r.get('Status', '').startswith('ERROR')])}")

# # Display results summary
# retrain_df = pd.DataFrame(retrain_results)
# print("\nRetraining Summary:")
# print(retrain_df[['Original_Dataset', 'Total_Samples', 'Status']])

# # Verify files were replaced
# print(f"\n=== VERIFICATION ===")
# for result in retrain_results:
#     if result.get('Status') == 'RETRAINED':
#         chemnet_file = os.path.join(chemnet_folder, f"{result['ChemNet_Dataset']}.pkl")
#         if os.path.exists(chemnet_file):
#             file_time = os.path.getmtime(chemnet_file)
#             print(f"{result['ChemNet_Dataset']}.pkl - Modified: {pd.Timestamp.fromtimestamp(file_time)}")
#         else:
#             print(f"{result['ChemNet_Dataset']}.pkl - FILE NOT FOUND")

## Random Forest on ChemNet

In [ ]:
import os
import gc
import pickle

# Load ChemNet datasets folder path
chemnet_folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/chemnet_grid_search_dataframes"

# Get all .pkl files in the folder
chemnet_pkl_files = [f for f in os.listdir(chemnet_folder) if f.endswith('.pkl')]
chemnet_dataset_names = [f.replace('.pkl', '') for f in chemnet_pkl_files]

print(f"Found {len(chemnet_dataset_names)} ChemNet datasets to process")

# Verify we have the right count
chemnet_thresh0_datasets = [name for name in chemnet_dataset_names if 'thresh_zero' in name]
chemnet_thresholded_datasets = [name for name in chemnet_dataset_names if 'thresh_zero' not in name]

print(f"  - ChemNet datasets with thresh0 (no threshold): {len(chemnet_thresh0_datasets)}")
print(f"  - ChemNet datasets with thresholds applied: {len(chemnet_thresholded_datasets)}")

# Initialize storage for ChemNet results
chemnet_results_r2 = []
chemnet_results_percent_error = []

# Dictionary to store individual errors for histogram analysis
saved_chemnet_errors = {}

# Process ChemNet datasets ONE AT A TIME (memory efficient)
for i, dataset_name in enumerate(sorted(chemnet_dataset_names), 1):
    print(f"Processing {i}/{len(chemnet_dataset_names)}: {dataset_name}")
    
    try:
        # Load only the current ChemNet dataset
        file_path = os.path.join(chemnet_folder, f"{dataset_name}.pkl")
        df = pd.read_pickle(file_path)
        
        # Prepare features and target (embedding columns are the features)
        feature_cols = [col for col in df.columns if col.startswith('emb_')]
        X = df[feature_cols]
        y = df['log_response']
        
        # Remove rows with NaN values
        valid_mask = ~(X.isna().any(axis=1) | y.isna())
        X_clean = X[valid_mask]
        y_clean = y[valid_mask]
        
        if len(X_clean) < 10:  # Skip if too few samples
            print(f"  Skipping {dataset_name}: Only {len(X_clean)} valid samples")
            continue
            
        # Split the data
        X_train, X_test, y_train, y_test = train_test_split(
            X_clean, y_clean, test_size=0.2, random_state=47
        )
        
        # Train Random Forest with limited CPU usage
        rf_chemnet = RandomForestRegressor(n_estimators=100, random_state=47, n_jobs=2)
        rf_chemnet.fit(X_train, y_train)

        # Make predictions
        y_train_pred = rf_chemnet.predict(X_train)
        y_test_pred = rf_chemnet.predict(X_test)
        
        # Calculate R² metrics
        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)
        
        # Calculate absolute percent error (undo log transform first)
        y_train_true_response = np.exp(y_train)
        y_train_pred_response = np.exp(y_train_pred)
        y_test_true_response = np.exp(y_test)
        y_test_pred_response = np.exp(y_test_pred)
        
        # Calculate individual errors for test set
        individual_errors = np.abs((y_test_pred_response - y_test_true_response) / y_test_true_response) * 100
        
        # Save individual errors for histogram analysis
        saved_chemnet_errors[dataset_name] = individual_errors
        
        # Calculate median and mean absolute percent error
        train_median_percent_error = 100 * (np.median(np.abs(y_train_pred_response - y_train_true_response) / y_train_true_response))
        test_median_percent_error = 100 * (np.median(np.abs(y_test_pred_response - y_test_true_response) / y_test_true_response))
        train_mean_percent_error = 100 * (np.mean(np.abs(y_train_pred_response - y_train_true_response) / y_train_true_response))
        test_mean_percent_error = 100 * (np.mean(np.abs(y_test_pred_response - y_test_true_response) / y_test_true_response))

        # Store results
        chemnet_results_r2.append({
            'Dataset': dataset_name,
            'Train_R2': train_r2,
            'Test_R2': test_r2,
            'Samples': len(X_clean),
            'Features': len(feature_cols)
        })
        
        chemnet_results_percent_error.append({
            'Dataset': dataset_name,
            'Train_Median_Percent_Error': train_median_percent_error,
            'Test_Median_Percent_Error': test_median_percent_error,
            'Train_Mean_Percent_Error': train_mean_percent_error,
            'Test_Mean_Percent_Error': test_mean_percent_error,
            'Samples': len(X_clean),
            'Features': len(feature_cols)
        })
        
        print(f"Completed: Test R² = {test_r2:.4f}, Test Median % Error = {test_median_percent_error:.1f}%")
        
    except Exception as e:
        print(f"Error processing {dataset_name}: {str(e)}")
        continue
    
    finally:
        # Always clean up memory after each dataset
        if 'df' in locals():
            del df
        if 'X' in locals():
            del X, y, X_clean, y_clean
        if 'rf_chemnet' in locals():
            del rf_chemnet
        gc.collect()
        
        # Periodic deeper cleanup every 20 datasets
        if i % 20 == 0:
            print(f"  Deep cleanup after {i} datasets...")
            gc.collect()

# Convert results to DataFrames
df_chemnet_r2_results = pd.DataFrame(chemnet_results_r2)
df_chemnet_percent_error_results = pd.DataFrame(chemnet_results_percent_error)

print(f"\nCompleted! Processed {len(chemnet_results_r2)} ChemNet datasets successfully.")
print(f"Saved individual errors for {len(saved_chemnet_errors)} datasets")
print(f"Results stored in: df_chemnet_r2_results, df_chemnet_percent_error_results")

# Display summary statistics
print("\n=== CHEMNET SUMMARY STATISTICS ===")
print("Test R² Statistics:")
print(df_chemnet_r2_results['Test_R2'].describe())

print("\nTest Median Percent Error Statistics:")
print(df_chemnet_percent_error_results['Test_Median_Percent_Error'].describe())

print("\nTest Mean Percent Error Statistics:")
print(df_chemnet_percent_error_results['Test_Mean_Percent_Error'].describe())

# Show top 10 performing ChemNet datasets by Test R²
print("\n=== TOP 10 CHEMNET DATASETS BY TEST R² ===")
top_chemnet_r2 = df_chemnet_r2_results.nlargest(10, 'Test_R2')[['Dataset', 'Test_R2', 'Features']]
print(top_chemnet_r2.to_string(index=False))

# Show comparison between thresh0 (no threshold) and thresholded ChemNet datasets
print("\n=== CHEMNET THRESH0 vs THRESHOLDED COMPARISON ===")
chemnet_thresh0_results = df_chemnet_r2_results[df_chemnet_r2_results['Dataset'].str.contains('thresh_zero')]
chemnet_thresholded_results = df_chemnet_r2_results[~df_chemnet_r2_results['Dataset'].str.contains('thresh_zero')]

if len(chemnet_thresh0_results) > 0:
    print(f"ChemNet thresh0 datasets (no threshold) - Mean Test R²: {chemnet_thresh0_results['Test_R2'].mean():.4f}")
    print(f"Best ChemNet thresh0 dataset: {chemnet_thresh0_results.loc[chemnet_thresh0_results['Test_R2'].idxmax(), 'Dataset']} (R² = {chemnet_thresh0_results['Test_R2'].max():.4f})")

if len(chemnet_thresholded_results) > 0:
    print(f"ChemNet thresholded datasets - Mean Test R²: {chemnet_thresholded_results['Test_R2'].mean():.4f}")
    print(f"Best ChemNet thresholded dataset: {chemnet_thresholded_results.loc[chemnet_thresholded_results['Test_R2'].idxmax(), 'Dataset']} (R² = {chemnet_thresholded_results['Test_R2'].max():.4f})")

In [ ]:
# import os
# import pickle

# # Load all ChemNet datasets from the chemnet_grid_search_dataframes folder
# chemnet_folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/chemnet_grid_search_dataframes"

# # Get all .pkl files in the folder
# chemnet_pkl_files = [f for f in os.listdir(chemnet_folder) if f.endswith('.pkl')]
# chemnet_dataset_names = [f.replace('.pkl', '') for f in chemnet_pkl_files]

# print(f"Found {len(chemnet_dataset_names)} ChemNet datasets to process")

# # Load ChemNet datasets into globals for compatibility with existing code
# for dataset_name in chemnet_dataset_names:
#     file_path = os.path.join(chemnet_folder, f"{dataset_name}.pkl")
#     globals()[dataset_name] = pd.read_pickle(file_path)

# # Verify we have the right count
# chemnet_thresh0_datasets = [name for name in chemnet_dataset_names if 'thresh_zero' in name]
# chemnet_thresholded_datasets = [name for name in chemnet_dataset_names if 'thresh_zero' not in name]

# print(f"  - ChemNet datasets with thresh0 (no threshold): {len(chemnet_thresh0_datasets)}")
# print(f"  - ChemNet datasets with thresholds applied: {len(chemnet_thresholded_datasets)}")

# # Initialize storage for ChemNet results
# chemnet_results_r2 = []
# chemnet_results_percent_error = []

# # Dictionary to store individual errors for histogram analysis
# saved_chemnet_errors = {}

# # Loop through the ChemNet datasets
# for i, dataset_name in enumerate(sorted(chemnet_dataset_names), 1):
#     print(f"Processing {i}/{len(chemnet_dataset_names)}: {dataset_name}")
    
#     try:
#         # Fetch the dataset from globals()
#         df = globals()[dataset_name]
        
#         # Prepare features and target (embedding columns are the features)
#         feature_cols = [col for col in df.columns if col.startswith('emb_')]
#         X = df[feature_cols]
#         y = df['log_response']
        
#         # Remove rows with NaN values
#         valid_mask = ~(X.isna().any(axis=1) | y.isna())
#         X_clean = X[valid_mask]
#         y_clean = y[valid_mask]
        
#         if len(X_clean) < 10:  # Skip if too few samples
#             print(f"  Skipping {dataset_name}: Only {len(X_clean)} valid samples")
#             continue
            
#         # Split the data
#         X_train, X_test, y_train, y_test = train_test_split(
#             X_clean, y_clean, test_size=0.2, random_state=47
#         )
        
#         # Train Random Forest
#         rf_chemnet = RandomForestRegressor(n_estimators=100, random_state=47, n_jobs=2)
#         rf_chemnet.fit(X_train, y_train)

#         # Make predictions
#         y_train_pred = rf_chemnet.predict(X_train)
#         y_test_pred = rf_chemnet.predict(X_test)
        
#         # Calculate R² metrics
#         train_r2 = r2_score(y_train, y_train_pred)
#         test_r2 = r2_score(y_test, y_test_pred)
        
#         # Calculate absolute percent error (undo log transform first)
#         y_train_true_response = np.exp(y_train)
#         y_train_pred_response = np.exp(y_train_pred)
#         y_test_true_response = np.exp(y_test)
#         y_test_pred_response = np.exp(y_test_pred)
        
#         # Calculate individual errors for test set
#         individual_errors = np.abs((y_test_pred_response - y_test_true_response) / y_test_true_response) * 100
        
#         # Save individual errors for histogram analysis
#         saved_chemnet_errors[dataset_name] = individual_errors
        
#         # Calculate median and mean absolute percent error
#         train_median_percent_error = 100 * (np.median(np.abs(y_train_pred_response - y_train_true_response) / y_train_true_response))
#         test_median_percent_error = 100 * (np.median(np.abs(y_test_pred_response - y_test_true_response) / y_test_true_response))
#         train_mean_percent_error = 100 * (np.mean(np.abs(y_train_pred_response - y_train_true_response) / y_train_true_response))
#         test_mean_percent_error = 100 * (np.mean(np.abs(y_test_pred_response - y_test_true_response) / y_test_true_response))

#         # Store results
#         chemnet_results_r2.append({
#             'Dataset': dataset_name,
#             'Train_R2': train_r2,
#             'Test_R2': test_r2,
#             'Samples': len(X_clean),
#             'Features': len(feature_cols)
#         })
        
#         chemnet_results_percent_error.append({
#             'Dataset': dataset_name,
#             'Train_Median_Percent_Error': train_median_percent_error,
#             'Test_Median_Percent_Error': test_median_percent_error,
#             'Train_Mean_Percent_Error': train_mean_percent_error,
#             'Test_Mean_Percent_Error': test_mean_percent_error,
#             'Samples': len(X_clean),
#             'Features': len(feature_cols)
#         })
        
#         print(f"Completed: Test R² = {test_r2:.4f}, Test Median % Error = {test_median_percent_error:.1f}%")
        
#     except Exception as e:
#         print(f"Error processing {dataset_name}: {str(e)}")
#         continue

# # Convert results to DataFrames
# df_chemnet_r2_results = pd.DataFrame(chemnet_results_r2)
# df_chemnet_percent_error_results = pd.DataFrame(chemnet_results_percent_error)

# print(f"\nCompleted! Processed {len(chemnet_results_r2)} ChemNet datasets successfully.")
# print(f"Saved individual errors for {len(saved_chemnet_errors)} datasets")
# print(f"Results stored in: df_chemnet_r2_results, df_chemnet_percent_error_results")


## ChemNet Heatmap

In [ ]:
# Create the actual heatmaps for ChemNet visualization
# First, let's extract bin size and threshold from the ChemNet dataset names and add to results
def parse_chemnet_dataset_name(dataset_name):
    """Extract bin size and threshold from ChemNet dataset name"""
    # Remove 'chemnet_emb_' prefix
    name_part = dataset_name.replace('chemnet_emb_', '')
    
    # Handle thresh_zero case (no threshold)
    if 'thresh_zero' in name_part:
        # Extract bin size
        bin_part = name_part.split('_thresh_zero')[0].replace('bin', '')
        bin_size = float(bin_part.replace('_', '.'))
        threshold = 0.0
    else:
        # Extract bin size and threshold
        parts = name_part.split('_thresh')
        bin_part = parts[0].replace('bin', '')
        bin_size = float(bin_part.replace('_', '.'))
        
        thresh_part = parts[1].split('_df3_QQpos_spectra')[0]
        threshold = float(thresh_part.replace('_', '.'))
    
    return bin_size, threshold

# Add bin_size and threshold columns to ChemNet results DataFrames
for df_results in [df_chemnet_r2_results, df_chemnet_percent_error_results]:
    bin_sizes = []
    thresholds = []
    
    for dataset_name in df_results['Dataset']:
        bin_size, threshold = parse_chemnet_dataset_name(dataset_name)
        bin_sizes.append(bin_size)
        thresholds.append(threshold)
    
    df_results['BinSize'] = bin_sizes
    df_results['Threshold'] = thresholds

# Check for and remove duplicates before creating pivot tables
print("Checking for duplicates in ChemNet results...")
print(f"Original df_chemnet_r2_results shape: {df_chemnet_r2_results.shape}")

# Remove duplicates based on BinSize + Threshold combination (keep first occurrence)
df_chemnet_r2_results = df_chemnet_r2_results.drop_duplicates(subset=['BinSize', 'Threshold'], keep='first')
df_chemnet_percent_error_results = df_chemnet_percent_error_results.drop_duplicates(subset=['BinSize', 'Threshold'], keep='first')

print(f"After removing duplicates: {df_chemnet_r2_results.shape}")

# Now create pivot tables for ChemNet
chemnet_r2_pivot = df_chemnet_r2_results.pivot(index='BinSize', columns='Threshold', values='Test_R2') 
chemnet_median_percent_error_pivot = df_chemnet_percent_error_results.pivot(index='BinSize', columns='Threshold', values='Test_Median_Percent_Error')
chemnet_mean_percent_error_pivot = df_chemnet_percent_error_results.pivot(index='BinSize', columns='Threshold', values='Test_Mean_Percent_Error')

# List all expected thresholds
thresholds_subset = [0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 50, 100]
bins_subset = [0.05, 0.1, 0.5, 1, 2, 5, 10, 25, 50, 100, 200, 500, 1000]

# Reindex pivot tables to show all columns, filling missing with NaN
chemnet_r2_pivot = chemnet_r2_pivot.reindex(columns=thresholds_subset, index=bins_subset)
chemnet_median_percent_error_pivot = chemnet_median_percent_error_pivot.reindex(columns=thresholds_subset, index=bins_subset)
chemnet_mean_percent_error_pivot = chemnet_mean_percent_error_pivot.reindex(columns=thresholds_subset, index=bins_subset)


# Also create individual larger heatmaps for ChemNet for better detail
def create_detailed_heatmap_chemnet(pivot_data, metric_name, cmap, figsize=(12, 8), vmin=None, vmax=None):
    """Create a detailed heatmap for a single ChemNet metric"""
    plt.figure(figsize=figsize)
    
    # Create heatmap
    sns.heatmap(pivot_data, 
                annot=True, 
                fmt='.3f' if 'R²' in metric_name else '.1f', 
                cmap=cmap,
                square=False,
                linewidths=0.5,
                vmin=vmin,
                vmax=vmax,
                cbar_kws={'label': f'Test {metric_name}', 'shrink': 0.8})
    
    plt.title(f'ChemNet: {metric_name} by Bin Size and Threshold', fontsize=16, fontweight='bold')
    plt.xlabel('Threshold Value', fontsize=14)
    plt.ylabel('Bin Size', fontsize=14)
    plt.gca().invert_yaxis()
    
    # Improve readability
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    
    # Add text annotation for best performance
    if 'R²' in metric_name:
        best_val = pivot_data.max().max()
        plt.text(0.02, 0.98, f'Best R²: {best_val:.4f}', 
                transform=plt.gca().transAxes, 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                verticalalignment='top')
    else:
        best_val = pivot_data.min().min()
        plt.text(0.02, 0.98, f'Best {metric_name}: {best_val:.1f}%', 
                transform=plt.gca().transAxes, 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                verticalalignment='top')
    
    plt.tight_layout()
    plt.savefig(f"/home/dlipsey/MITLincolnLabs/Figures/ChemNet_{metric_name}_by_Bin_Size_and_Threshold")

    plt.show()

# Create detailed individual ChemNet heatmaps
print("Creating detailed ChemNet heatmaps...")

# create_detailed_heatmap_chemnet(chemnet_r2_pivot, 'R²', 'RdYlBu')     
create_detailed_heatmap_chemnet(chemnet_median_percent_error_pivot, 'Median_Absolute_Percent_Error', 'RdYlBu_r', vmin=1.0, vmax=100.0) 
create_detailed_heatmap_chemnet(chemnet_mean_percent_error_pivot, 'Mean_Absolute_Percent_Error', 'RdYlBu_r', vmin=1.0, vmax=100.0)

## ChemNet Histograms (No re-prediction)

In [ ]:
# Create histograms using saved ChemNet errors
target_combinations = [
    (0.1, 0.1),    # Bin 0.1, Threshold 0.1
    (1000, 5),     # Bin 1000, Threshold 5
    (10, 5),       # Bin 10, Threshold 5
    (10, 0.01),     # Bin 10, Threshold 0.01
    (0.05, 0.001),  # Bin 0.05, Threshold 0.001
    (0.5, 0.01),   # Bin 0.5, Threshold 0.01
    (1000, 0.1),   # Bin 1000, Threshold 0.1
    (100, 0.01)   # Bin 100, Threshold 0.01
]


# Function to create dataset name from bin size and threshold
def create_chemnet_dataset_name(bin_size, threshold):
    """Create ChemNet dataset name from bin size and threshold"""
    bin_str = str(bin_size).replace('.', '_')
    if threshold == 0:
        thresh_str = "zero"
    else:
        thresh_str = str(threshold).replace('.', '_')
    return f"chemnet_emb_bin{bin_str}_thresh{thresh_str}_df3_QQpos_spectra"

def create_spectral_dataset_name(bin_size, threshold):
    """Create Spectral dataset name from bin size and threshold"""
    bin_str = str(bin_size).replace('.', '_')
    if threshold == 0:
        thresh_str = "zero"
    else:
        thresh_str = str(threshold).replace('.', '_')
    return f"bin{bin_str}_thresh{thresh_str}_df3_QQpos_spectra"

# Collect prediction errors for each combination using saved data
chemnet_errors_list = []
spectral_errors_list = []

for bin_size, threshold in target_combinations:
    # Get dataset names
    chemnet_dataset_name = create_chemnet_dataset_name(bin_size, threshold)
    spectral_dataset_name = create_spectral_dataset_name(bin_size, threshold)
    
    # Get ChemNet errors from saved data
    if chemnet_dataset_name in saved_chemnet_errors:
        chemnet_errors_list.append(saved_chemnet_errors[chemnet_dataset_name])
        print(f"Found ChemNet errors for {chemnet_dataset_name}")
    else:
        chemnet_errors_list.append(np.array([]))
        print(f"No ChemNet data found for {chemnet_dataset_name}")
    
    # Get Spectral errors from saved data
    if spectral_dataset_name in saved_spectral_errors:
        spectral_errors_list.append(saved_spectral_errors[spectral_dataset_name])
        print(f"Found Spectral errors for {spectral_dataset_name}")
    else:
        spectral_errors_list.append(np.array([]))
        print(f"No Spectral data found for {spectral_dataset_name}")

# Create side-by-side 2x4 grids
fig, axes = plt.subplots(4, 4, figsize=(16, 16))

# ChemNet histograms (left 2 columns)
# Spectral histograms (right 2 columns)

# Plot ChemNet error histograms
for i, (bin_size, threshold) in enumerate(target_combinations):
    row = i // 2
    col = i % 2
    ax = axes[row, col]  # Left 2x2 grid
    errors = chemnet_errors_list[i]
    
    if len(errors) > 0:
        ax.hist(errors, bins=20, alpha=0.7, color='lightblue', 
                edgecolor='black', linewidth=1)
        
        # Add median line
        median_error = np.median(errors)
        ax.axvline(median_error, color='red', linestyle='--', linewidth=2)
        
        ax.text(0.7, 0.9, f'Median: {median_error:.1f}%', 
               transform=ax.transAxes, fontsize=9,
               bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
        
        ax.set_title(f'ChemNet Errors\nBin {bin_size}, Thresh {threshold}', fontsize=10)
        ax.set_ylabel('Frequency', fontsize=9)
        ax.set_xlabel('Absolute % Error', fontsize=9)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'No ChemNet data\nBin {bin_size}\nThresh {threshold}', 
               ha='center', va='center', transform=ax.transAxes)

# Plot Spectral error histograms
for i, (bin_size, threshold) in enumerate(target_combinations):
    row = i // 2
    col = (i % 2) + 2  
    ax = axes[row, col]
    errors = spectral_errors_list[i]
    
    if len(errors) > 0:
        ax.hist(errors, bins=20, alpha=0.7, color='lightcoral', 
                edgecolor='black', linewidth=1)
        
        # Add median line
        median_error = np.median(errors)
        ax.axvline(median_error, color='darkred', linestyle='--', linewidth=2)
        
        ax.text(0.7, 0.9, f'Median: {median_error:.1f}%', 
               transform=ax.transAxes, fontsize=9,
               bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
        
        ax.set_title(f'Spectral Errors\nBin {bin_size}, Thresh {threshold}', fontsize=10)
        ax.set_ylabel('Frequency', fontsize=9)
        ax.set_xlabel('Absolute % Error', fontsize=9)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'No Spectral data\nBin {bin_size}\nThresh {threshold}', 
               ha='center', va='center', transform=ax.transAxes)

plt.suptitle('Prediction Error Distributions: ChemNet vs Spectral Random Forests', fontsize=14)
plt.tight_layout()
plt.show()

# Print summary of comparison
print("\n=== COMPARISON SUMMARY ===")
for i, (bin_size, threshold) in enumerate(target_combinations):
    chemnet_errors = chemnet_errors_list[i]
    spectral_errors = spectral_errors_list[i]
    
    print(f"\nBin {bin_size}, Threshold {threshold}:")
    
    if len(chemnet_errors) > 0:
        chemnet_median = np.median(chemnet_errors)
        print(f"  ChemNet median error: {chemnet_median:.1f}%")
    else:
        print(f"  ChemNet: No data available")
    
    if len(spectral_errors) > 0:
        spectral_median = np.median(spectral_errors)
        print(f"  Spectral median error: {spectral_median:.1f}%")
    else:
        print(f"  Spectral: No data available")
    
    if len(chemnet_errors) > 0 and len(spectral_errors) > 0:
        if chemnet_median < spectral_median:
            print(f"  → ChemNet performs better by {spectral_median - chemnet_median:.1f}%")
        else:
            print(f"  → Spectral performs better by {chemnet_median - spectral_median:.1f}%")

## Histograms

In [ ]:
# HISTOGRAM OF PREDICTION ERRORS: ChemNet vs Spectral Random Forests (Loading from Files)
target_combinations = [
    (0.1, 0.1),    # Bin 0.1, Threshold 0.1
    (1000, 5),     # Bin 1000, Threshold 5
    (10, 5),       # Bin 10, Threshold 5
    (10, 0.01),     # Bin 10, Threshold 0.01
    (0.05, 0.001),  # Bin 0.05, Threshold 0.001
    (0.5, 0.01),   # Bin 0.5, Threshold 0.01
    (1000, 0.1),   # Bin 1000, Threshold 0.1
    (100, 0.01)   # Bin 100, Threshold 0.01
]

# Function to get dataset name from bin/threshold
def get_dataset_name(bin_size, threshold):
    if threshold == 0.0:
        return f"bin{str(bin_size).replace('.', '_')}_thresh_zero_df3_QQpos_spectra"
    else:
        return f"bin{str(bin_size).replace('.', '_')}_thresh{str(threshold).replace('.', '_')}_df3_QQpos_spectra"

# Function to get ChemNet dataset name
def get_chemnet_dataset_name(bin_size, threshold):
    original_name = get_dataset_name(bin_size, threshold)
    return f"chemnet_emb_{original_name}"

# Define folders
grid_search_folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/grid_search_dataframes"
chemnet_folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/chemnet_grid_search_dataframes"

# Collect prediction errors for each combination
chemnet_errors_list = []
spectral_errors_list = []

for bin_size, threshold in target_combinations:
    # Get dataset names
    spectral_dataset_name = get_dataset_name(bin_size, threshold)
    chemnet_dataset_name = get_chemnet_dataset_name(bin_size, threshold)
    
    # Load and process spectral data
    spectral_file_path = os.path.join(grid_search_folder, f"{spectral_dataset_name}.pkl")
    if os.path.exists(spectral_file_path):
        spectral_df = pd.read_pickle(spectral_file_path)
        
        # Same preprocessing as in your RF training
        feature_cols = [col for col in spectral_df.columns if col not in ['SMILES_spectra', 'Response', 'log_response', 'index_id']]
        X_spec = spectral_df[feature_cols]
        y_spec = spectral_df['log_response']
        
        # Remove NaN values
        valid_mask = ~(X_spec.isna().any(axis=1) | y_spec.isna())
        X_spec_clean = X_spec[valid_mask]
        y_spec_clean = y_spec[valid_mask]
        
        if len(X_spec_clean) > 10:  # Only proceed if enough samples
            # Train/test split
            X_train, X_test, y_train, y_test = train_test_split(
                X_spec_clean, y_spec_clean, test_size=0.2, random_state=42
            )
            
            # Train RF and get individual prediction errors
            rf_spec = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=2)
            rf_spec.fit(X_train, y_train)
            y_pred = rf_spec.predict(X_test)
            
            # Calculate individual absolute percent errors
            y_test_response = np.exp(y_test)
            y_pred_response = np.exp(y_pred)
            individual_errors = np.abs((y_test_response - y_pred_response) / y_test_response) * 100
            spectral_errors_list.append(individual_errors)
        else:
            spectral_errors_list.append(np.array([]))
    else:
        print(f"Spectral dataset not found: {spectral_file_path}")
        spectral_errors_list.append(np.array([]))
    
    # Load and process ChemNet data
    chemnet_file_path = os.path.join(chemnet_folder, f"{chemnet_dataset_name}.pkl")
    if os.path.exists(chemnet_file_path):
        chemnet_df = pd.read_pickle(chemnet_file_path)
        
        # Same preprocessing as in your ChemNet RF training
        feature_cols = [col for col in chemnet_df.columns if col.startswith('emb_')]
        X_chem = chemnet_df[feature_cols]
        y_chem = chemnet_df['log_response']
        
        # Remove NaN values
        valid_mask = ~(X_chem.isna().any(axis=1) | y_chem.isna())
        X_chem_clean = X_chem[valid_mask]
        y_chem_clean = y_chem[valid_mask]
        
        if len(X_chem_clean) > 10:  # Only proceed if enough samples
            # Train/test split
            X_train, X_test, y_train, y_test = train_test_split(
                X_chem_clean, y_chem_clean, test_size=0.2, random_state=42
            )
            
            # Train RF and get individual prediction errors
            rf_chem = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=2)
            rf_chem.fit(X_train, y_train)
            y_pred = rf_chem.predict(X_test)
            
            # Calculate individual absolute percent errors
            y_test_response = np.exp(y_test)
            y_pred_response = np.exp(y_pred)
            individual_errors = np.abs((y_test_response - y_pred_response) / y_test_response) * 100
            chemnet_errors_list.append(individual_errors)
        else:
            chemnet_errors_list.append(np.array([]))
    else:
        print(f"ChemNet dataset not found: {chemnet_file_path}")
        chemnet_errors_list.append(np.array([]))

# Create side-by-side 2x2 grids
fig, axes = plt.subplots(4, 4, figsize=(16, 10))

# ChemNet histograms (left 2 columns)
# Spectral histograms (right 2 columns)

# Plot ChemNet error histograms
for i, (bin_size, threshold) in enumerate(target_combinations):
    row = i // 2
    col = i % 2
    ax = axes[row, col]  # Left 2x2 grid
    errors = chemnet_errors_list[i]
    
    if len(errors) > 0:
        ax.hist(errors, bins=20, alpha=0.7, color='lightblue', 
                edgecolor='black', linewidth=1)
        
        # Add median line
        median_error = np.median(errors)
        ax.axvline(median_error, color='red', linestyle='--', linewidth=2)
        
        ax.text(0.7, 0.9, f'Median: {median_error:.1f}%', 
               transform=ax.transAxes, fontsize=9,
               bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
        
        ax.set_title(f'ChemNet Errors\nBin {bin_size}, Thresh {threshold}', fontsize=10)
        ax.set_ylabel('Frequency', fontsize=9)
        ax.set_xlabel('Absolute % Error', fontsize=9)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'No ChemNet data\nBin {bin_size}\nThresh {threshold}', 
               ha='center', va='center', transform=ax.transAxes)

# Plot Spectral error histograms
for i, (bin_size, threshold) in enumerate(target_combinations):
    row = i // 2
    col = (i % 2) + 2  
    ax = axes[row, col]
    errors = spectral_errors_list[i]
    
    if len(errors) > 0:
        ax.hist(errors, bins=20, alpha=0.7, color='lightcoral', 
                edgecolor='black', linewidth=1)
        
        # Add median line
        median_error = np.median(errors)
        ax.axvline(median_error, color='darkred', linestyle='--', linewidth=2)
        
        ax.text(0.7, 0.9, f'Median: {median_error:.1f}%', 
               transform=ax.transAxes, fontsize=9,
               bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
        
        ax.set_title(f'Spectral Errors\nBin {bin_size}, Thresh {threshold}', fontsize=10)
        ax.set_ylabel('Frequency', fontsize=9)
        ax.set_xlabel('Absolute % Error', fontsize=9)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'No Spectral data\nBin {bin_size}\nThresh {threshold}', 
               ha='center', va='center', transform=ax.transAxes)

plt.suptitle('Prediction Error Distributions: ChemNet vs Spectral Random Forests', fontsize=14)
plt.tight_layout()
plt.show()

# Print summary of comparison
print("\n=== COMPARISON SUMMARY ===")
for i, (bin_size, threshold) in enumerate(target_combinations):
    chemnet_errors = chemnet_errors_list[i]
    spectral_errors = spectral_errors_list[i]
    
    print(f"\nBin {bin_size}, Threshold {threshold}:")
    
    if len(chemnet_errors) > 0:
        chemnet_median = np.median(chemnet_errors)
        print(f"  ChemNet median error: {chemnet_median:.1f}%")
    else:
        print(f"  ChemNet: No data available")
    
    if len(spectral_errors) > 0:
        spectral_median = np.median(spectral_errors)
        print(f"  Spectral median error: {spectral_median:.1f}%")
    else:
        print(f"  Spectral: No data available")
    
    if len(chemnet_errors) > 0 and len(spectral_errors) > 0:
        if chemnet_median < spectral_median:
            print(f"  → ChemNet performs better by {spectral_median - chemnet_median:.1f}%")
        else:
            print(f"  → Spectral performs better by {chemnet_median - spectral_median:.1f}%")

In [ ]:
# PLOT HISTOGRAMS OF TOP 5% ERRORS ONLY
target_combinations = [
    (0.1, 0.1),    # Bin 0.1, Threshold 0.1
    (1000, 5),     # Bin 1000, Threshold 5
    (10, 5),       # Bin 10, Threshold 5
    (10, 0.01),     # Bin 10, Threshold 0.01
    (0.05, 0.001),  # Bin 0.05, Threshold 0.001
    (0.5, 0.01),   # Bin 0.5, Threshold 0.01
    (1000, 0.1),   # Bin 1000, Threshold 0.1
    (100, 0.01)   # Bin 100, Threshold 0.01
]

# Function to get dataset names
def get_dataset_name(bin_size, threshold):
    if threshold == 0.0:
        return f"bin{str(bin_size).replace('.', '_')}_thresh_zero_df3_QQpos_spectra"
    else:
        return f"bin{str(bin_size).replace('.', '_')}_thresh{str(threshold).replace('.', '_')}_df3_QQpos_spectra"

def get_chemnet_dataset_name(bin_size, threshold):
    original_name = get_dataset_name(bin_size, threshold)
    return f"chemnet_emb_{original_name}"

# Define folders
grid_search_folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/grid_search_dataframes"
chemnet_folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/chemnet_grid_search_dataframes"

# Collect top 5% errors for each combination
chemnet_top5_errors = []
spectral_top5_errors = []

for bin_size, threshold in target_combinations:
    spectral_dataset_name = get_dataset_name(bin_size, threshold)
    chemnet_dataset_name = get_chemnet_dataset_name(bin_size, threshold)
    
    # Process spectral data
    spectral_file_path = os.path.join(grid_search_folder, f"{spectral_dataset_name}.pkl")
    if os.path.exists(spectral_file_path):
        spectral_df = pd.read_pickle(spectral_file_path)
        feature_cols = [col for col in spectral_df.columns if col not in ['SMILES_spectra', 'Response', 'log_response', 'index_id']]
        X_spec = spectral_df[feature_cols]
        y_spec = spectral_df['log_response']
        
        valid_mask = ~(X_spec.isna().any(axis=1) | y_spec.isna())
        X_spec_clean = X_spec[valid_mask]
        y_spec_clean = y_spec[valid_mask]
        
        if len(X_spec_clean) > 10:
            X_train, X_test, y_train, y_test = train_test_split(X_spec_clean, y_spec_clean, test_size=0.2, random_state=47)
            rf_spec = RandomForestRegressor(n_estimators=100, random_state=47, n_jobs=2)
            rf_spec.fit(X_train, y_train)
            y_pred = rf_spec.predict(X_test)
            
            y_test_response = np.exp(y_test)
            y_pred_response = np.exp(y_pred)
            individual_errors = np.abs((y_test_response - y_pred_response) / y_test_response) * 100
            
            # Get top 5% errors
            top_5_threshold = np.percentile(individual_errors, 95)
            top_5_errors = individual_errors[individual_errors >= top_5_threshold]
            spectral_top5_errors.append(top_5_errors)
        else:
            spectral_top5_errors.append(np.array([]))
    else:
        spectral_top5_errors.append(np.array([]))
    
    # Process ChemNet data
    chemnet_file_path = os.path.join(chemnet_folder, f"{chemnet_dataset_name}.pkl")
    if os.path.exists(chemnet_file_path):
        chemnet_df = pd.read_pickle(chemnet_file_path)
        feature_cols = [col for col in chemnet_df.columns if col.startswith('emb_')]
        X_chem = chemnet_df[feature_cols]
        y_chem = chemnet_df['log_response']
        
        valid_mask = ~(X_chem.isna().any(axis=1) | y_chem.isna())
        X_chem_clean = X_chem[valid_mask]
        y_chem_clean = y_chem[valid_mask]
        
        if len(X_chem_clean) > 10:
            X_train, X_test, y_train, y_test = train_test_split(X_chem_clean, y_chem_clean, test_size=0.2, random_state=47)
            rf_chem = RandomForestRegressor(n_estimators=100, random_state=47, n_jobs=2)
            rf_chem.fit(X_train, y_train)
            y_pred = rf_chem.predict(X_test)
            
            y_test_response = np.exp(y_test)
            y_pred_response = np.exp(y_pred)
            individual_errors = np.abs((y_test_response - y_pred_response) / y_test_response) * 100
            
            # Get top 5% errors
            top_5_threshold = np.percentile(individual_errors, 95)
            top_5_errors = individual_errors[individual_errors >= top_5_threshold]
            chemnet_top5_errors.append(top_5_errors)
        else:
            chemnet_top5_errors.append(np.array([]))
    else:
        chemnet_top5_errors.append(np.array([]))

# Calculate global min/max for standardized scales
all_chemnet_errors = np.concatenate([errors for errors in chemnet_top5_errors if len(errors) > 0])
all_spectral_errors = np.concatenate([errors for errors in spectral_top5_errors if len(errors) > 0])
all_errors = np.concatenate([all_chemnet_errors, all_spectral_errors])

# Set standardized x-axis limits
x_min = all_errors.min()
x_max = all_errors.max()
x_margin = (x_max - x_min) * 0.05  # 5% margin
x_limits = (x_min - x_margin, x_max + x_margin)

# Calculate max frequency for standardized y-axis
max_freq = 0
for errors in chemnet_top5_errors + spectral_top5_errors:
    if len(errors) > 0:
        hist_counts, _ = np.histogram(errors, bins=15)
        max_freq = max(max_freq, hist_counts.max())

y_limits = (0, max_freq * 1.1)  # 10% margin above max frequency

# Create histograms of top 5% errors only
fig, axes = plt.subplots(4, 4, figsize=(16, 10))

# Plot ChemNet top 5% error histograms
for i, (bin_size, threshold) in enumerate(target_combinations):
    row = i // 2
    col = i % 2
    ax = axes[row, col]
    errors = chemnet_top5_errors[i]
    
    if len(errors) > 0:
        ax.hist(errors, bins=15, alpha=0.7, color='lightblue', edgecolor='black', linewidth=1)
        ax.set_xlim(x_limits)
        ax.set_ylim(y_limits)
        median_error = np.median(errors)
        ax.axvline(median_error, color='red', linestyle='--', linewidth=2)
        
        ax.text(0.7, 0.9, f'Median: {median_error:.1f}%\nCount: {len(errors)}', 
               transform=ax.transAxes, fontsize=9,
               bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
        
        ax.set_title(f'ChemNet Top 5% Errors\nBin {bin_size}, Thresh {threshold}', fontsize=10)
        ax.set_ylabel('Frequency', fontsize=9)
        ax.set_xlabel('Absolute % Error', fontsize=9)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'No ChemNet data\nBin {bin_size}\nThresh {threshold}', 
               ha='center', va='center', transform=ax.transAxes)

# Plot Spectral top 5% error histograms
for i, (bin_size, threshold) in enumerate(target_combinations):
    row = i // 2
    col = (i % 2) + 2
    ax = axes[row, col]
    errors = spectral_top5_errors[i]
    
    if len(errors) > 0:
        ax.hist(errors, bins=15, alpha=0.7, color='lightcoral', edgecolor='black', linewidth=1)
        ax.set_xlim(x_limits)
        ax.set_ylim(y_limits)
        median_error = np.median(errors)
        ax.axvline(median_error, color='darkred', linestyle='--', linewidth=2)
        
        ax.text(0.7, 0.9, f'Median: {median_error:.1f}%\nCount: {len(errors)}', 
               transform=ax.transAxes, fontsize=9,
               bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
        
        ax.set_title(f'Spectral Top 5% Errors\nBin {bin_size}, Thresh {threshold}', fontsize=10)
        ax.set_ylabel('Frequency', fontsize=9)
        ax.set_xlabel('Absolute % Error', fontsize=9)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'No Spectral data\nBin {bin_size}\nThresh {threshold}', 
               ha='center', va='center', transform=ax.transAxes)

plt.suptitle('Top 5% Prediction Error Distributions: ChemNet vs Spectral Random Forests', fontsize=14)
plt.tight_layout()
plt.show()

# Print summary of top 5% errors
print("\n=== TOP 5% ERRORS SUMMARY ===")
for i, (bin_size, threshold) in enumerate(target_combinations):
    print(f"\nBin {bin_size}, Threshold {threshold}:")
    
    chemnet_errors = chemnet_top5_errors[i]
    spectral_errors = spectral_top5_errors[i]
    
    if len(chemnet_errors) > 0:
        print(f"  ChemNet top 5%: {len(chemnet_errors)} samples, median {np.median(chemnet_errors):.1f}%")
    else:
        print(f"  ChemNet: No data")
    
    if len(spectral_errors) > 0:
        print(f"  Spectral top 5%: {len(spectral_errors)} samples, median {np.median(spectral_errors):.1f}%")
    else:
        print(f"  Spectral: No data")

## Index IDs

In [ ]:
# Print index_ids for top 5% outliers from saved data
target_combinations = [
    (0.1, 0.1),    # Bin 0.1, Threshold 0.1
    (1000, 5),     # Bin 1000, Threshold 5
    (10, 5),       # Bin 10, Threshold 5
    (10, 0.01),     # Bin 10, Threshold 0.01
    (0.05, 0.001),  # Bin 0.05, Threshold 0.001
    (0.5, 0.01),   # Bin 0.5, Threshold 0.01
    (1000, 0.1),   # Bin 1000, Threshold 0.1
    (100, 0.01)   # Bin 100, Threshold 0.01
]
print("=== INDEX_IDS FOR TOP 5% OUTLIERS ===")

for bin_size, threshold in target_combinations:
    print(f"\n{'='*60}")
    print(f"BIN {bin_size}, THRESHOLD {threshold}")
    print(f"{'='*60}")
    
    # Get dataset names
    chemnet_dataset_name = create_chemnet_dataset_name(bin_size, threshold)
    spectral_dataset_name = create_spectral_dataset_name(bin_size, threshold)
    
    # Load and process ChemNet data to get index_ids
    chemnet_folder = "/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/chemnet_grid_search_dataframes"
    chemnet_file_path = os.path.join(chemnet_folder, f"{chemnet_dataset_name}.pkl")
    
    chemnet_outlier_ids = []
    if os.path.exists(chemnet_file_path):
        chemnet_df = pd.read_pickle(chemnet_file_path)
        feature_cols = [col for col in chemnet_df.columns if col.startswith('emb_')]
        X_chem = chemnet_df[feature_cols]
        y_chem = chemnet_df['log_response']
        
        valid_mask = ~(X_chem.isna().any(axis=1) | y_chem.isna())
        if valid_mask.sum() > 10:
            X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
                X_chem[valid_mask], y_chem[valid_mask], 
                chemnet_df.loc[valid_mask, 'index_id'], 
                test_size=0.2, random_state=47
            )
            
            # Use saved errors to identify top 5%
            if chemnet_dataset_name in saved_chemnet_errors:
                errors = saved_chemnet_errors[chemnet_dataset_name]
                top_5_threshold = np.percentile(errors, 95)
                top_5_mask = errors >= top_5_threshold
                # Convert boolean mask to indices and then get the corresponding index_ids
                outlier_indices = np.where(top_5_mask)[0]
                chemnet_outlier_ids = idx_test.iloc[outlier_indices].tolist()
    
    # Load and process Spectral data to get index_ids
    grid_search_folder = "/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/grid_search_dataframes"
    spectral_file_path = os.path.join(grid_search_folder, f"{spectral_dataset_name}.pkl")
    
    spectral_outlier_ids = []
    if os.path.exists(spectral_file_path):
        spectral_df = pd.read_pickle(spectral_file_path)
        feature_cols = [col for col in spectral_df.columns if col not in ['SMILES_spectra', 'Response', 'log_response', 'index_id']]
        X_spec = spectral_df[feature_cols]
        y_spec = spectral_df['log_response']
        
        valid_mask = ~(X_spec.isna().any(axis=1) | y_spec.isna())
        if valid_mask.sum() > 10:
            X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
                X_spec[valid_mask], y_spec[valid_mask], 
                spectral_df.loc[valid_mask, 'index_id'], 
                test_size=0.2, random_state=42
            )
            
            # Use saved errors to identify top 5%
            if spectral_dataset_name in saved_spectral_errors:
                errors = saved_spectral_errors[spectral_dataset_name]
                top_5_threshold = np.percentile(errors, 95)
                top_5_mask = errors >= top_5_threshold
                # Convert boolean mask to indices and then get the corresponding index_ids
                outlier_indices = np.where(top_5_mask)[0]
                spectral_outlier_ids = idx_test.iloc[outlier_indices].tolist()
    
    # Print results side by side for easy comparison
    print(f"ChemNet Top 5% Outlier IDs ({len(chemnet_outlier_ids)} samples):")
    if chemnet_outlier_ids:
        print(f"  {sorted(chemnet_outlier_ids)}")
    else:
        print("  No data available")
    
    print(f"\nSpectral Top 5% Outlier IDs ({len(spectral_outlier_ids)} samples):")
    if spectral_outlier_ids:
        print(f"  {sorted(spectral_outlier_ids)}")
    else:
        print("  No data available")
    
    # Find common outliers
    if chemnet_outlier_ids and spectral_outlier_ids:
        common_outliers = set(chemnet_outlier_ids) & set(spectral_outlier_ids)
        chemnet_only = set(chemnet_outlier_ids) - set(spectral_outlier_ids)
        spectral_only = set(spectral_outlier_ids) - set(chemnet_outlier_ids)
        
        print(f"\nCommon Outliers ({len(common_outliers)} samples):")
        if common_outliers:
            print(f"  {sorted(list(common_outliers))}")
        else:
            print("  None")
            
        print(f"\nChemNet-Only Outliers ({len(chemnet_only)} samples):")
        if chemnet_only:
            print(f"  {sorted(list(chemnet_only))}")
        else:
            print("  None")
            
        print(f"\nSpectral-Only Outliers ({len(spectral_only)} samples):")
        if spectral_only:
            print(f"  {sorted(list(spectral_only))}")
        else:
            print("  None")

print(f"\n{'='*60}")
print("ANALYSIS COMPLETE")
print(f"{'='*60}")

# Outlier Investigation

In [ ]:
# Complete list of (bin_size, threshold) combinations - All 169 combinations
combinations = [
    # Bin size 0.05 (13 combinations)
    (0.05, 0), (0.05, 0.001), (0.05, 0.005), (0.05, 0.01), (0.05, 0.05), (0.05, 0.1), (0.05, 0.5), 
    (0.05, 1), (0.05, 2), (0.05, 5), (0.05, 10), (0.05, 50), (0.05, 100),
    
    # Bin size 0.1 (13 combinations)
    (0.1, 0), (0.1, 0.001), (0.1, 0.005), (0.1, 0.01), (0.1, 0.05), (0.1, 0.1), (0.1, 0.5), 
    (0.1, 1), (0.1, 2), (0.1, 5), (0.1, 10), (0.1, 50), (0.1, 100),
    
    # Bin size 0.5 (13 combinations)
    (0.5, 0), (0.5, 0.001), (0.5, 0.005), (0.5, 0.01), (0.5, 0.05), (0.5, 0.1), (0.5, 0.5), 
    (0.5, 1), (0.5, 2), (0.5, 5), (0.5, 10), (0.5, 50), (0.5, 100),
    
    # Bin size 1 (13 combinations)
    (1, 0), (1, 0.001), (1, 0.005), (1, 0.01), (1, 0.05), (1, 0.1), (1, 0.5), 
    (1, 1), (1, 2), (1, 5), (1, 10), (1, 50), (1, 100),
    
    # Bin size 2 (13 combinations)
    (2, 0), (2, 0.001), (2, 0.005), (2, 0.01), (2, 0.05), (2, 0.1), (2, 0.5), 
    (2, 1), (2, 2), (2, 5), (2, 10), (2, 50), (2, 100),
    
    # Bin size 5 (13 combinations)
    (5, 0), (5, 0.001), (5, 0.005), (5, 0.01), (5, 0.05), (5, 0.1), (5, 0.5), 
    (5, 1), (5, 2), (5, 5), (5, 10), (5, 50), (5, 100),
    
    # Bin size 10 (13 combinations)
    (10, 0), (10, 0.001), (10, 0.005), (10, 0.01), (10, 0.05), (10, 0.1), (10, 0.5), 
    (10, 1), (10, 2), (10, 5), (10, 10), (10, 50), (10, 100),
    
    # Bin size 25 (13 combinations)
    (25, 0), (25, 0.001), (25, 0.005), (25, 0.01), (25, 0.05), (25, 0.1), (25, 0.5), 
    (25, 1), (25, 2), (25, 5), (25, 10), (25, 50), (25, 100),
    
    # Bin size 50 (13 combinations)
    (50, 0), (50, 0.001), (50, 0.005), (50, 0.01), (50, 0.05), (50, 0.1), (50, 0.5), 
    (50, 1), (50, 2), (50, 5), (50, 10), (50, 50), (50, 100),
    
    # Bin size 100 (13 combinations)
    (100, 0), (100, 0.001), (100, 0.005), (100, 0.01), (100, 0.05), (100, 0.1), (100, 0.5), 
    (100, 1), (100, 2), (100, 5), (100, 10), (100, 50), (100, 100),
    
    # Bin size 200 (13 combinations)
    (200, 0), (200, 0.001), (200, 0.005), (200, 0.01), (200, 0.05), (200, 0.1), (200, 0.5), 
    (200, 1), (200, 2), (200, 5), (200, 10), (200, 50), (200, 100),
    
    # Bin size 500 (13 combinations)
    (500, 0), (500, 0.001), (500, 0.005), (500, 0.01), (500, 0.05), (500, 0.1), (500, 0.5), 
    (500, 1), (500, 2), (500, 5), (500, 10), (500, 50), (500, 100),
    
    # Bin size 1000 (13 combinations)
    (1000, 0), (1000, 0.001), (1000, 0.005), (1000, 0.01), (1000, 0.05), (1000, 0.1), (1000, 0.5), 
    (1000, 1), (1000, 2), (1000, 5), (1000, 10), (1000, 50), (1000, 100)
]

print(f"Total combinations: {len(combinations)}")
print("Combinations:", combinations)
# Combination set
target_combinations = combinations  

print(f"=== CHRONIC OUTLIER ANALYSIS ACROSS {len(target_combinations)} COMBINATIONS ===")
print("Combinations:", target_combinations)

# Function to get all outlier IDs for a method across all combinations
def get_all_outlier_ids(target_combinations, method='chemnet'):
    """
    Get outlier IDs for all combinations
    method: 'chemnet' or 'spectral'
    """
    outlier_id_counts = {}
    successful_combinations = 0
    
    for bin_size, threshold in target_combinations:
        if method == 'chemnet':
            dataset_name = create_chemnet_dataset_name(bin_size, threshold)
            folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/chemnet_grid_search_dataframes"
            saved_errors = saved_chemnet_errors
            random_state = 47
            feature_filter = lambda cols: [col for col in cols if col.startswith('emb_')]
        else:  # spectral
            dataset_name = create_spectral_dataset_name(bin_size, threshold)
            folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/grid_search_dataframes"
            saved_errors = saved_spectral_errors
            random_state = 42
            feature_filter = lambda cols: [col for col in cols if col not in ['SMILES_spectra', 'Response', 'log_response', 'index_id']]
        
        file_path = os.path.join(folder, f"{dataset_name}.pkl")
        
        if os.path.exists(file_path):
            df = pd.read_pickle(file_path)
            feature_cols = feature_filter(df.columns)
            X = df[feature_cols]
            y = df['log_response']
            
            valid_mask = ~(X.isna().any(axis=1) | y.isna())
            if valid_mask.sum() > 10:
                X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
                    X[valid_mask], y[valid_mask], 
                    df.loc[valid_mask, 'index_id'], 
                    test_size=0.2, random_state=random_state
                )
                
                # Use saved errors to identify top 5%
                if dataset_name in saved_errors:
                    errors = saved_errors[dataset_name]
                    top_5_threshold = np.percentile(errors, 95)
                    top_5_mask = errors >= top_5_threshold
                    outlier_indices = np.where(top_5_mask)[0]
                    outlier_ids = idx_test.iloc[outlier_indices].tolist()
                    
                    successful_combinations += 1
                    
                    # Count occurrences of each ID
                    for idx_id in outlier_ids:
                        outlier_id_counts[idx_id] = outlier_id_counts.get(idx_id, 0) + 1
    
    return outlier_id_counts, successful_combinations

chemnet_counts, chemnet_successful = get_all_outlier_ids(target_combinations, 'chemnet')
spectral_counts, spectral_successful = get_all_outlier_ids(target_combinations, 'spectral')

# Analyze ChemNet chronic outliers
print("\n" + "="*60)
print("CHEMNET CHRONIC OUTLIERS")
print("="*60)

print(f"Successfully processed: {chemnet_successful}/{len(target_combinations)} combinations")
print(f"Total unique outlier index_ids: {len(chemnet_counts)}")

# Focus on chronic outliers (appearing in 5+ combinations)
chemnet_chronic = [(idx_id, count) for idx_id, count in chemnet_counts.items() if count >= 5]
chemnet_chronic_sorted = sorted(chemnet_chronic, key=lambda x: x[1], reverse=True)

if chemnet_chronic_sorted:
    print(f"\nCHRONIC OUTLIERS (appearing in 5+ combinations): {len(chemnet_chronic_sorted)}")
    for idx_id, count in chemnet_chronic_sorted:
        percentage = (count / chemnet_successful) * 100
        print(f"  index_id {idx_id}: {count}/{chemnet_successful} combinations ({percentage:.1f}%)")
    
    # Highlight the most chronic ones
    super_chronic = [x for x in chemnet_chronic_sorted if x[1] >= max(3, chemnet_successful // 2)]
    if super_chronic:
        print(f"\nSUPER CHRONIC OUTLIERS (≥{max(3, chemnet_successful // 2)} combinations): {len(super_chronic)}")
        for idx_id, count in super_chronic:
            percentage = (count / chemnet_successful) * 100
            print(f"index_id {idx_id}: {count}/{chemnet_successful} combinations ({percentage:.1f}%)")
else:
    print("\nNo chronic outliers found (no index_ids appear in 5+ combinations)")

# Analyze Spectral chronic outliers
print("\n" + "="*60)
print("SPECTRAL CHRONIC OUTLIERS")
print("="*60)

print(f"Successfully processed: {spectral_successful}/{len(target_combinations)} combinations")
print(f"Total unique outlier index_ids: {len(spectral_counts)}")

# Focus on chronic outliers (appearing in 5+ combinations)
spectral_chronic = [(idx_id, count) for idx_id, count in spectral_counts.items() if count >= 5]
spectral_chronic_sorted = sorted(spectral_chronic, key=lambda x: x[1], reverse=True)

if spectral_chronic_sorted:
    print(f"\nCHRONIC OUTLIERS (appearing in 5+ combinations): {len(spectral_chronic_sorted)}")
    for idx_id, count in spectral_chronic_sorted:
        percentage = (count / spectral_successful) * 100
        print(f"index_id {idx_id}: {count}/{spectral_successful} combinations ({percentage:.1f}%)")
    
    # Highlight the most chronic ones
    super_chronic = [x for x in spectral_chronic_sorted if x[1] >= max(3, spectral_successful // 2)]
    if super_chronic:
        print(f"\nSUPER CHRONIC OUTLIERS (≥{max(3, spectral_successful // 2)} combinations): {len(super_chronic)}")
        for idx_id, count in super_chronic:
            percentage = (count / spectral_successful) * 100
            print(f"index_id {idx_id}: {count}/{spectral_successful} combinations ({percentage:.1f}%)")
else:
    print("\nNo chronic outliers found (no index_ids appear in 5+ combinations)")

# Summary
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"ChemNet chronic outliers: {len(chemnet_chronic_sorted)}")
print(f"Spectral chronic outliers: {len(spectral_chronic_sorted)}")
print("\nThese index_ids represent spectra that are consistently difficult to predict")
print("across multiple bin/threshold combinations within each method.")
print(f"{'='*60}")

In [ ]:
# Efficient combined chronic outlier analysis - Response and Log Response distributions
df1 = df3_QQpos_spectra
df2 = add_response_and_log_response(df3_QQpos, df3_QQpos)

# Extract chronic outlier index_ids and get DataFrame indices
chronic_chemnet_outlier_ids = [idx_id for idx_id, count in chemnet_chronic_sorted]
chronic_spectra_outlier_ids = [idx_id for idx_id, count in spectral_chronic_sorted]
chemnet_outlier_indices = df1[df1['index_id'].isin(chronic_chemnet_outlier_ids)].index
spectra_outlier_indices = df1[df1['index_id'].isin(chronic_spectra_outlier_ids)].index

# Extract response data
chemnet_responses = df2.loc[chemnet_outlier_indices, 'Response']
spectra_responses = df2.loc[spectra_outlier_indices, 'Response']
chemnet_log_responses = df2.loc[chemnet_outlier_indices, 'log_response']
spectra_log_responses = df2.loc[spectra_outlier_indices, 'log_response']

# Calculate standardized axis limits for log response plots
all_log_responses = np.concatenate([chemnet_log_responses, spectra_log_responses])
x_min, x_max = all_log_responses.min(), all_log_responses.max()
x_margin = (x_max - x_min) * 0.05
x_limits = (x_min - x_margin, x_max + x_margin)

bins = 20
chemnet_counts, _ = np.histogram(chemnet_log_responses, bins=bins)
spectra_counts, _ = np.histogram(spectra_log_responses, bins=bins)
max_count = max(chemnet_counts.max(), spectra_counts.max())
y_limits = (0, max_count * 1.1)

# Create Response distribution plots
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(chemnet_responses, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
plt.title('ChemNet Chronic Outliers\nResponse Value Distribution')
plt.xlabel('Response')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(spectra_responses, bins=20, color='salmon', edgecolor='black', alpha=0.7)
plt.title('Spectra Chronic Outliers\nResponse Value Distribution')
plt.xlabel('Response')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/home/dlipsey/MITLincolnLabs/Figures/Chronic_chemnet_and_spectra_outliers_response")
plt.show()

# Create Log Response distribution plots
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(chemnet_log_responses, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
plt.title('ChemNet Chronic Outliers\nLog Response Value Distribution')
plt.xlabel('Log Response')
plt.ylabel('Frequency')
plt.xlim(x_limits)
plt.ylim(y_limits)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(spectra_log_responses, bins=20, color='salmon', edgecolor='black', alpha=0.7)
plt.title('Spectra Chronic Outliers\nLog Response Value Distribution')
plt.xlabel('Log Response')
plt.ylabel('Frequency')
plt.xlim(x_limits)
plt.ylim(y_limits)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/home/dlipsey/MITLincolnLabs/Figures/Chronic_chemnet_and_spectra_outliers_log_response")
plt.show()

In [ ]:
# Bin size and threshold size
bin_size = '0_1'        # Format like dataset the dataset name strings
threshold = '0_5'       # Ditto
# =====================================================================

# Load the true ChemNet embeddings for SMILES matching
true_chemnet_path = '/home/dlipsey/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QQpos_no_repeats.csv'
true_chemnet_df = pd.read_csv(true_chemnet_path)

# Construct dataset name from bin size and threshold
dataset_name = f'chemnet_emb_bin{bin_size}_thresh{threshold}_df3_QQpos_spectra'

# Load the dataset from the pickle file
chemnet_folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/chemnet_grid_search_dataframes"
chemnet_file_path = os.path.join(chemnet_folder, f"{dataset_name}.pkl")
chemnet_data = pd.read_pickle(chemnet_file_path)

# Get embedding columns (exclude SMILES and Response columns)
# Filter out non-numeric columns more carefully
numeric_cols = chemnet_data.select_dtypes(include=[np.number]).columns.tolist()
emb_cols = [col for col in numeric_cols if col not in ['Response']]

# Get the true embedding columns to match dimensions
true_emb_cols = [col for col in true_chemnet_df.columns if col != 'SMILES']
n_true_features = len(true_emb_cols)

# Ensure we only use the same number of features as the true embeddings
emb_cols_matched = emb_cols[:n_true_features]  

print(f"Dataset embedding columns: {len(emb_cols)}")
print(f"True embedding columns: {len(true_emb_cols)}")
print(f"Using {len(emb_cols_matched)} matched columns for PCA")

# Get the embedding data with matched dimensions
X_chemnet = chemnet_data[emb_cols_matched].values

# Create chronic outlier labels
chemnet_ids = chemnet_data.index.tolist()
is_chronic_outlier = [idx in chronic_chemnet_outlier_ids for idx in chemnet_ids]

# Perform PCA
pca = PCA(n_components=2)
X_chemnet_pca = pca.fit_transform(X_chemnet)

# Find true embeddings by matching SMILES
true_embeddings_pca = []
true_smiles_found = []

for idx, row in chemnet_data.iterrows():
    smiles = row['SMILES_spectra']
    # Find matching SMILES in true ChemNet data
    true_match = true_chemnet_df[true_chemnet_df['SMILES'] == smiles]
    
    if not true_match.empty:
        # Get the true embedding columns (exclude SMILES and any other non-embedding columns)
        true_emb_cols = [col for col in true_chemnet_df.columns if col != 'SMILES']
        true_embedding = true_match[true_emb_cols].values[0]
        
        # Transform the true embedding using the same PCA
        true_embedding_pca = pca.transform(true_embedding.reshape(1, -1))[0]
        true_embeddings_pca.append(true_embedding_pca)
        true_smiles_found.append(smiles)

true_embeddings_pca = np.array(true_embeddings_pca)

# Create the plot
plt.figure(figsize=(12, 8))

# Plot non-chronic outliers in blue
non_chronic_mask = ~np.array(is_chronic_outlier)
plt.scatter(X_chemnet_pca[non_chronic_mask, 0], 
           X_chemnet_pca[non_chronic_mask, 1],
           c='blue', marker='o', alpha=0.7, s=50, label='Non-chronic outliers')

# Plot chronic outliers in red
chronic_mask = np.array(is_chronic_outlier)
plt.scatter(X_chemnet_pca[chronic_mask, 0], 
           X_chemnet_pca[chronic_mask, 1],
           c='red', marker='o', alpha=0.7, s=50, label='Chronic outliers')

# Plot true embeddings as black squares
if len(true_embeddings_pca) > 0:
    plt.scatter(true_embeddings_pca[:, 0], true_embeddings_pca[:, 1], 
               c='black', marker='s', s=25, label=f'True embeddings ({len(true_embeddings_pca)})', 
               linewidth=2)

# Customize the plot
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title(f'2D PCA of ChemNet Embeddings: {dataset_name}\nColored by Chronic Outlier Status')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Print summary statistics
print(f"Dataset: {dataset_name}")
print(f"Total samples: {len(chemnet_data)}")
print(f"Chronic outliers: {sum(is_chronic_outlier)}")
print(f"Non-chronic outliers: {sum(non_chronic_mask)}")
print(f"True embeddings found: {len(true_embeddings_pca)}")
print(f"PC1 explains {pca.explained_variance_ratio_[0]:.1%} of variance")
print(f"PC2 explains {pca.explained_variance_ratio_[1]:.1%} of variance")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.1%}")

plt.savefig(f"/home/dlipsey/MITLincolnLabs/Figures/bin{bin_size}_thresh{threshold}_outlier_PCA.png")
plt.show()

# Expanded dataframe

In [ ]:
# # Configuration for the three bin/threshold combinations
# combinations = [
#     ('0_05', '0_001'),  # (0.05, 0.001)
#     ('10', '10'),       # (10, 10)  
#     ('0_05', '5')      # (0.05, 5)
# ]

# # Initialize results dictionary
# all_results = {}

# # Load the original df3_QQpos dataframe
# df3_QQpos_original = df3_QQpos.copy()

# # Add index_id from df3_QQpos_spectra to df3_QQpos_original if it's missing
# if 'index_id' not in df3_QQpos_original.columns:
#     df3_QQpos_original['index_id'] = df3_QQpos_spectra['index_id']
#     print(f"Added index_id column to df3_QQpos_original from df3_QQpos_spectra")

# for bin_size, threshold in combinations:
#     print(f"Processing bin={bin_size}, threshold={threshold}")
    
#     # Construct file paths
#     spec_path = f'/home/dlipsey/MITLincolnLabs/MIT_LL_data/grid_search_dataframes/bin{bin_size}_thresh{threshold}_df3_QQpos_spectra.pkl'
#     chemnet_path = f'/home/dlipsey/MITLincolnLabs/MIT_LL_data/chemnet_grid_search_dataframes/chemnet_emb_bin{bin_size}_thresh{threshold}_df3_QQpos_spectra.pkl'
    
#     # Load datasets
#     try:
#         spec_data = pd.read_pickle(spec_path)
#         chemnet_data = pd.read_pickle(chemnet_path)
        
#         # Debug: Check for index_id column
#         print(f"  Spec data columns: {list(spec_data.columns)}")
#         print(f"  ChemNet data columns: {list(chemnet_data.columns)}")
        
#         if 'index_id' not in spec_data.columns:
#             print(f"  WARNING: 'index_id' not found in spec_data. Available columns: {list(spec_data.columns)}")
#             continue
#         if 'index_id' not in chemnet_data.columns:
#             print(f"  WARNING: 'index_id' not found in chemnet_data. Available columns: {list(chemnet_data.columns)}")
#             continue
            
#     except FileNotFoundError as e:
#         print(f"Dataset not found: {e}, skipping...")
#         continue
    
#     # Prepare spectra data for Random Forest
#     X_spec_cols = [col for col in spec_data.columns if col not in ['SMILES_spectra', 'index_id', 'Response', 'log_response']]
#     X_spec = spec_data[X_spec_cols]
#     y_spec = spec_data['Response']
#     spec_indices = spec_data['index_id']
    
#     # Prepare ChemNet data for Random Forest  
#     X_chem_cols = [col for col in chemnet_data.columns if col not in ['SMILES_spectra', 'index_id', 'Response', 'log_response']]
#     X_chem = chemnet_data[X_chem_cols]
#     y_chem = chemnet_data['Response']
#     chem_indices = chemnet_data['index_id']
    
#     # Use matching seeds and stratify by index_id to ensure consistent test sets
#     # Train Spectra Random Forest
#     X_train_spec, X_test_spec, y_train_spec, y_test_spec, idx_train_spec, idx_test_spec = train_test_split(
#         X_spec, y_spec, spec_indices, test_size=0.2, random_state=42
#     )
#     rf_spec = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=2)
#     rf_spec.fit(X_train_spec, y_train_spec)
    
#     # Get spectra predictions for all data
#     spec_predictions = rf_spec.predict(X_spec)
#     spec_errors = np.abs((spec_predictions - y_spec) / y_spec) * 100
    
#     # Create test set indicator for spectra
#     spec_test_set = spec_indices.isin(idx_test_spec).astype(int)
    
#     # Train ChemNet Random Forest with same random seed
#     X_train_chem, X_test_chem, y_train_chem, y_test_chem, idx_train_chem, idx_test_chem = train_test_split(
#         X_chem, y_chem, chem_indices, test_size=0.2, random_state=42
#     )
#     rf_chemnet = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=2)
#     rf_chemnet.fit(X_train_chem, y_train_chem)
    
#     # Get ChemNet predictions for all data
#     chemnet_predictions = rf_chemnet.predict(X_chem)
#     chemnet_errors = np.abs((chemnet_predictions - y_chem) / y_chem) * 100
    
#     # Create test set indicator for chemnet
#     chem_test_set = chem_indices.isin(idx_test_chem).astype(int)
    
#     # Create results DataFrames with index_id and test_set indicator
#     spec_results = pd.DataFrame({
#         'index_id': spec_indices,
#         f'spec_pred_bin{bin_size}_thresh{threshold}': spec_predictions,
#         f'spec_error_bin{bin_size}_thresh{threshold}': spec_errors,
#         f'spec_test_set_bin{bin_size}_thresh{threshold}': spec_test_set
#     })
    
#     chemnet_results = pd.DataFrame({
#         'index_id': chem_indices,
#         f'chemnet_pred_bin{bin_size}_thresh{threshold}': chemnet_predictions,
#         f'chemnet_error_bin{bin_size}_thresh{threshold}': chemnet_errors,
#         f'chemnet_test_set_bin{bin_size}_thresh{threshold}': chem_test_set
#     })
    
#     # Store results
#     all_results[f'spec_{bin_size}_{threshold}'] = spec_results
#     all_results[f'chemnet_{bin_size}_{threshold}'] = chemnet_results
    
#     print(f"Completed bin={bin_size}, threshold={threshold}")
#     print(f"  Spectra predictions: {len(spec_predictions)}")
#     print(f"  ChemNet predictions: {len(chemnet_predictions)}")
#     print(f"  Spectra test set size: {spec_test_set.sum()}")
#     print(f"  ChemNet test set size: {chem_test_set.sum()}")

# # Check if df3_QQpos_original has index_id column
# print(f"\nOriginal df3_QQpos columns: {list(df3_QQpos_original.columns)}")
# if 'index_id' not in df3_QQpos_original.columns:
#     print("ERROR: 'index_id' not found in df3_QQpos_original")
#     # Try alternative column names
#     possible_index_cols = [col for col in df3_QQpos_original.columns if 'index' in col.lower()]
#     print(f"Possible index columns: {possible_index_cols}")
#     if possible_index_cols:
#         print(f"Using '{possible_index_cols[0]}' as index column")
#         df3_QQpos_original = df3_QQpos_original.rename(columns={possible_index_cols[0]: 'index_id'})

# # Merge all results with the original df3_QQpos dataframe
# final_df = df3_QQpos_original.copy()

# for key, results_df in all_results.items():
#     print(f"Merging {key}...")
#     print(f"  Results df columns: {list(results_df.columns)}")
#     print(f"  Final df columns before merge: {list(final_df.columns)}")
    
#     final_df = final_df.merge(results_df, on='index_id', how='left')
#     print(f"Merged {key}: {len(results_df)} rows")

# # Save the final dataframe
# output_path = '/home/dlipsey/MITLincolnLabs/MIT_LL_data/df3_QQpos_with_RF_predictions.csv'
# final_df.to_csv(output_path, index=False)
# print(f"Saved final dataframe with {len(final_df)} rows to {output_path}")
# print(f"New columns added: {len(final_df.columns) - len(df3_QQpos_original.columns)}")

In [ ]:
# df = pd.read_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/df3_QQpos_with_RF_predictions.csv')
# print(f"Loaded dataframe with shape: {df.shape}")
# print(f"Columns: {list(df.columns)}")
# print("\nFirst few rows:")
# print(df.head())
# df.head()


In [ ]:
# Load the pickle file
df = pd.read_pickle('/home/dlipsey/MITLincolnLabs/MIT_LL_data/cond_enc_outputs/cond_enc_bin0_05_thresh0_1_df3_QQpos_spectra.pkl')

# Display basic info about the dataframe
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Show the first few rows
df.head()

# Conditional Encoder Outlier Investigation

In [ ]:
# This uses the saved Conditional Encoder Output dataframes, whichever training method is their source to make grid search heatmaps
# and then to make the outlier histograms that we also want.


cond_enc_folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/cond_enc_outputs"
cond_enc_files = [f for f in os.listdir(cond_enc_folder) if f.endswith('.pkl') and f.startswith('cond_enc_')]

# Storage for conditional encoder results
cond_encoder_results = []

for i, file_name in enumerate(sorted(cond_enc_files), 1):
    try:
        # Load conditional encoder outputs
        file_path = os.path.join(cond_enc_folder, file_name)
        cond_df = pd.read_pickle(file_path)
        
        # Extract dataset name (remove 'cond_enc_' prefix and '.pkl' suffix)
        dataset_name = file_name.replace('cond_enc_', '').replace('.pkl', '')
        
        # Check if required columns exist
        if 'cond_tox_pred' not in cond_df.columns or 'log_response' not in cond_df.columns:
            continue
        
        # Split data into train/test (same logic as your conditional encoder training)
        # Use SMILES grouping to ensure proper split
        counts = cond_df['SMILES_spectra'].value_counts()
        valid_smiles = counts[counts >= 4].index
        filtered_df = cond_df[cond_df['SMILES_spectra'].isin(valid_smiles)].copy()
        
        if len(filtered_df) < 20:
            continue
        
        # Create train/test split by SMILES
        train_indices = []
        test_indices = []
        
        np.random.seed(42)
        for smiles, group in filtered_df.groupby('SMILES_spectra'):
            idx = group.index.tolist()
            n = len(idx)
            np.random.shuffle(idx)
            split = n // 2
            test_idx = idx[:split]
            train_idx = idx[split:]
            train_indices.extend(train_idx)
            test_indices.extend(test_idx)
        
        train_data = filtered_df.loc[train_indices]
        test_data = filtered_df.loc[test_indices]
        
        # Get predictions and true values
        train_pred = train_data['cond_tox_pred'].values
        train_true = train_data['log_response'].values
        test_pred = test_data['cond_tox_pred'].values
        test_true = test_data['log_response'].values
        
        # Calculate percent errors (undo log transform to get back to original response scale)
        train_response_true = np.exp(train_true)
        train_response_pred = np.exp(train_pred)
        test_response_true = np.exp(test_true)
        test_response_pred = np.exp(test_pred)
        
        # Calculate absolute percent errors
        train_median_percent_error = 100 * np.median(np.abs(train_response_pred - train_response_true) / train_response_true)
        test_median_percent_error = 100 * np.median(np.abs(test_response_pred - test_response_true) / test_response_true)
        train_mean_percent_error = 100 * np.mean(np.abs(train_response_pred - train_response_true) / train_response_true)
        test_mean_percent_error = 100 * np.mean(np.abs(test_response_pred - test_response_true) / test_response_true)
        
        # Store results
        cond_encoder_results.append({
            'Dataset': dataset_name,
            'Train_Median_Percent_Error': train_median_percent_error,
            'Test_Median_Percent_Error': test_median_percent_error,
            'Train_Mean_Percent_Error': train_mean_percent_error,
            'Test_Mean_Percent_Error': test_mean_percent_error,
            'Samples': len(filtered_df),
            'Train_Samples': len(train_data),
            'Test_Samples': len(test_data)
        })
        
    except Exception as e:
        continue

# Convert to DataFrame
df_cond_percent_error_results = pd.DataFrame(cond_encoder_results)

# Parse dataset names to extract bin sizes and thresholds
def parse_cond_dataset_name(dataset_name):
    """Extract bin size and threshold from dataset name"""
    if 'thresh_zero' in dataset_name:
        bin_part = dataset_name.split('_thresh_zero')[0].replace('bin', '')
        bin_size = float(bin_part.replace('_', '.'))
        threshold = 0.0
    else:
        parts = dataset_name.split('_thresh')
        bin_part = parts[0].replace('bin', '')
        bin_size = float(bin_part.replace('_', '.'))
        
        thresh_part = parts[1].split('_df3_QQpos_spectra')[0]
        threshold = float(thresh_part.replace('_', '.'))
    
    return bin_size, threshold

# Add bin size and threshold columns
bin_sizes = []
thresholds = []

for dataset_name in df_cond_percent_error_results['Dataset']:
    bin_size, threshold = parse_cond_dataset_name(dataset_name)
    bin_sizes.append(bin_size)
    thresholds.append(threshold)

df_cond_percent_error_results['BinSize'] = bin_sizes
df_cond_percent_error_results['Threshold'] = thresholds

# Create pivot tables for heatmaps
thresholds_subset = [0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 50, 100]
bins_subset = [0.05, 0.1, 0.5, 1, 2, 5, 10, 25, 50, 100, 200, 500, 1000]

cond_median_error_pivot = df_cond_percent_error_results.pivot(index='BinSize', columns='Threshold', values='Test_Median_Percent_Error')
cond_mean_error_pivot = df_cond_percent_error_results.pivot(index='BinSize', columns='Threshold', values='Test_Mean_Percent_Error')

# Reindex to show all combinations
cond_median_error_pivot = cond_median_error_pivot.reindex(columns=thresholds_subset, index=bins_subset)
cond_mean_error_pivot = cond_mean_error_pivot.reindex(columns=thresholds_subset, index=bins_subset)

def create_detailed_heatmap_cond_enc(pivot_data, metric_name, cmap, figsize=(12, 8), vmin=None, vmax=None):
    """Create a detailed heatmap for a single conditional encoder metric"""
    plt.figure(figsize=figsize)
    
    # Create heatmap
    sns.heatmap(pivot_data, 
                annot=True, 
                fmt='.1f', 
                cmap=cmap,
                square=False,
                linewidths=0.5,
                vmin=vmin,
                vmax=vmax,
                cbar_kws={'label': f'Test {metric_name}', 'shrink': 0.8})
    
    plt.title(f'Conditional Encoder: {metric_name} by Bin Size and Threshold', fontsize=16, fontweight='bold')
    plt.xlabel('Threshold Value', fontsize=14)
    plt.ylabel('Bin Size', fontsize=14)
    plt.gca().invert_yaxis()
    
    # Improve readability
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    
    # Add text annotation for best performance
    best_val = pivot_data.min().min()
    plt.text(0.02, 0.98, f'Best {metric_name}: {best_val:.1f}%', 
            transform=plt.gca().transAxes, 
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
            verticalalignment='top')
    
    plt.tight_layout()
    plt.savefig(f"/home/dlipsey/MITLincolnLabs/Figures/Conditional_encoder_{metric_name}_by_Bin_Size_and_Threshold")
    plt.show()

# Create detailed individual heatmaps
create_detailed_heatmap_cond_enc(cond_median_error_pivot, 'Median_Absolute_Percent_Error', 'RdYlBu_r', vmin=0, vmax=100) 
create_detailed_heatmap_cond_enc(cond_mean_error_pivot, 'Mean_Absolute_Percent_Error', 'RdYlBu_r', vmin=0, vmax=100)

best_median_error_idx = df_cond_percent_error_results['Test_Median_Percent_Error'].idxmin()
best_median_error_dataset = df_cond_percent_error_results.loc[best_median_error_idx, 'Dataset']
best_median_error_value = df_cond_percent_error_results.loc[best_median_error_idx, 'Test_Median_Percent_Error']

best_mean_error_idx = df_cond_percent_error_results['Test_Mean_Percent_Error'].idxmin()
best_mean_error_dataset = df_cond_percent_error_results.loc[best_mean_error_idx, 'Dataset']
best_mean_error_value = df_cond_percent_error_results.loc[best_mean_error_idx, 'Test_Mean_Percent_Error']

In [ ]:
# Debug the duplicate entries issue
print("=== DEBUGGING CONDITIONAL ENCODER PIVOT TABLE ===")
print(f"Total rows in df_cond_percent_error_results: {len(df_cond_percent_error_results)}")
print(f"Columns: {df_cond_percent_error_results.columns.tolist()}")

# Check the first few rows
print("\nFirst 5 rows:")
print(df_cond_percent_error_results.head())

# Check for duplicates in BinSize and Threshold combinations
print("\n=== CHECKING FOR DUPLICATES ===")
duplicate_check = df_cond_percent_error_results.groupby(['BinSize', 'Threshold']).size()
duplicates = duplicate_check[duplicate_check > 1]

if len(duplicates) > 0:
    print(f"Found {len(duplicates)} duplicate BinSize/Threshold combinations:")
    for (bin_size, threshold), count in duplicates.items():
        print(f"  Bin {bin_size}, Threshold {threshold}: {count} entries")
    
    # Show the actual duplicate rows
    print("\nDuplicate rows:")
    for (bin_size, threshold), count in duplicates.items():
        print(f"\n--- Bin {bin_size}, Threshold {threshold} ---")
        duplicate_rows = df_cond_percent_error_results[
            (df_cond_percent_error_results['BinSize'] == bin_size) & 
            (df_cond_percent_error_results['Threshold'] == threshold)
        ]
        print(duplicate_rows[['Dataset', 'BinSize', 'Threshold', 'Test_Median_Percent_Error']])
else:
    print("No duplicate BinSize/Threshold combinations found")

# Check if there are any issues with the dataset name parsing
print("\n=== DATASET NAME PARSING CHECK ===")
print("Unique dataset names and their parsed values:")
for dataset_name in sorted(df_cond_percent_error_results['Dataset'].unique()):
    try:
        bin_size, threshold = parse_cond_dataset_name(dataset_name)
        print(f"  {dataset_name} -> Bin: {bin_size}, Threshold: {threshold}")
    except Exception as e:
        print(f"  {dataset_name} -> ERROR: {e}")

# Check the file names that were processed
print(f"\n=== FILES PROCESSED ===")
print(f"Total files found: {len(cond_enc_files)}")
print("Files processed:")
for file_name in sorted(cond_enc_files)[:10]:  # Show first 10
    print(f"  {file_name}")
if len(cond_enc_files) > 10:
    print(f"  ... and {len(cond_enc_files) - 10} more")

In [ ]:
# Load conditional encoder outputs and find chronic outliers
cond_enc_folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/cond_enc_outputs"
cond_enc_files = [f for f in os.listdir(cond_enc_folder) if f.endswith('.pkl') and f.startswith('cond_enc_')]

# Function to get all outlier IDs for conditional encoder across all combinations
def get_all_cond_outlier_ids(cond_enc_files):
    """Get outlier IDs for all conditional encoder combinations"""
    outlier_id_counts = {}
    successful_combinations = 0
    
    for file_name in cond_enc_files:
        try:
            file_path = os.path.join(cond_enc_folder, file_name)
            cond_df = pd.read_pickle(file_path)
            
            # Check if required columns exist
            if 'cond_tox_pred' not in cond_df.columns or 'log_response' not in cond_df.columns:
                continue
            
            # Split data into train/test (same logic as conditional encoder training)
            counts = cond_df['SMILES_spectra'].value_counts()
            valid_smiles = counts[counts >= 4].index
            filtered_df = cond_df[cond_df['SMILES_spectra'].isin(valid_smiles)].copy()
            
            if len(filtered_df) < 20:
                continue
            
            # Create train/test split by SMILES
            train_indices = []
            test_indices = []
            
            np.random.seed(42)
            for smiles, group in filtered_df.groupby('SMILES_spectra'):
                idx = group.index.tolist()
                n = len(idx)
                np.random.shuffle(idx)
                split = n // 2
                test_idx = idx[:split]
                train_idx = idx[split:]
                train_indices.extend(train_idx)
                test_indices.extend(test_idx)
            
            test_data = filtered_df.loc[test_indices]
            
            # Get predictions and true values
            test_pred = test_data['cond_tox_pred'].values
            test_true = test_data['log_response'].values
            
            # Calculate percent errors (undo log transform)
            test_response_true = np.exp(test_true)
            test_response_pred = np.exp(test_pred)
            
            # Calculate individual absolute percent errors
            individual_errors = np.abs((test_response_pred - test_response_true) / test_response_true) * 100
            
            # Get top 5% errors
            top_5_threshold = np.percentile(individual_errors, 95)
            top_5_mask = individual_errors >= top_5_threshold
            outlier_indices = np.where(top_5_mask)[0]
            outlier_ids = test_data.iloc[outlier_indices]['index_id'].tolist()
            
            successful_combinations += 1
            
            # Count occurrences of each ID
            for idx_id in outlier_ids:
                outlier_id_counts[idx_id] = outlier_id_counts.get(idx_id, 0) + 1
                
        except Exception as e:
            continue
    
    return outlier_id_counts, successful_combinations

# Get conditional encoder chronic outliers
cond_counts, cond_successful = get_all_cond_outlier_ids(cond_enc_files)

# Focus on chronic outliers (appearing in 5+ combinations)
cond_chronic = [(idx_id, count) for idx_id, count in cond_counts.items() if count >= 5]
cond_chronic_sorted = sorted(cond_chronic, key=lambda x: x[1], reverse=True)

# Extract chronic outlier index_ids
chronic_cond_outlier_ids = [idx_id for idx_id, count in cond_chronic_sorted]

# Get DataFrame indices for chronic outliers
df1 = df3_QQpos_spectra
df2 = df3_QQpos
df2 = add_response_and_log_response(df2, df3_QQpos)

cond_outlier_indices = df1[df1['index_id'].isin(chronic_cond_outlier_ids)].index

# Extract responses for chronic outliers
cond_responses = df2.loc[cond_outlier_indices, 'Response']

# Plot Response distribution
plt.figure(figsize=(6, 5))
plt.hist(cond_responses, bins=20, color='lightgreen', edgecolor='black', alpha=0.7)
plt.title('Conditional Encoder Chronic Outliers\nResponse Value Distribution')
plt.xlabel('Response')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/home/dlipsey/MITLincolnLabs/Figures/Chronic_cond_enc_outliers_response")
plt.show()

# Extract log responses for chronic outliers
cond_log_responses = df2.loc[cond_outlier_indices, 'log_response']

# Plot Log Response distribution
plt.figure(figsize=(6, 5))
plt.hist(cond_log_responses, bins=20, color='lightgreen', edgecolor='black', alpha=0.7)
plt.title('Conditional Encoder Chronic Outliers\nLog Response Value Distribution')
plt.xlabel('Log Response')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/home/dlipsey/MITLincolnLabs/Figures/Chronic_cond_enc_outliers_log_response")
plt.show()

# Plots

In [ ]:
# Configuration - easily changeable bin sizes
selected_bin_sizes = [0.05, 2, 10]  # Change these values as needed

# Extract data for the selected bin sizes
def extract_data_for_bin_sizes(results_df, bin_sizes):
    """Extract median percent error data for selected bin sizes"""
    data = {}
    for bin_size in bin_sizes:
        bin_data = results_df[results_df['BinSize'] == bin_size].copy()
        if not bin_data.empty:
            # Sort by threshold for proper line plotting
            bin_data = bin_data.sort_values('Threshold')
            data[bin_size] = {
                'thresholds': bin_data['Threshold'].values,
                'median_errors': bin_data['Test_Median_Percent_Error'].values
            }
    return data

# Extract data for all three methods
spectra_data = extract_data_for_bin_sizes(df_percent_error_results, selected_bin_sizes)
chemnet_data = extract_data_for_bin_sizes(df_chemnet_percent_error_results, selected_bin_sizes)
cond_enc_data = extract_data_for_bin_sizes(df_cond_percent_error_results, selected_bin_sizes)

# Create 3x3 grid of line plots
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Median Absolute Percent Error vs Threshold Values (Line Plots)', fontsize=16, fontweight='bold')

methods = ['Spectra', 'ChemNet', 'Conditional Encoder']
data_sources = [spectra_data, chemnet_data, cond_enc_data]
colors = ['blue', 'red', 'green']

for method_idx, (method_name, method_data, color) in enumerate(zip(methods, data_sources, colors)):
    for bin_idx, bin_size in enumerate(selected_bin_sizes):
        ax = axes[method_idx, bin_idx]
        
        if bin_size in method_data:
            thresholds = method_data[bin_size]['thresholds']
            errors = method_data[bin_size]['median_errors']
            
            # Plot line with markers
            ax.plot(thresholds, errors, marker='o', linewidth=2, markersize=6, 
                   color=color, alpha=0.7, label=f'Bin {bin_size}')
            
            # Add grid
            ax.grid(True, alpha=0.3)
            
            # Set labels
            if method_idx == 2:  # Bottom row
                ax.set_xlabel('Threshold Value', fontsize=10)
            if bin_idx == 0:  # Left column
                ax.set_ylabel('Median Absolute % Error', fontsize=10)
            
            # Set title
            ax.set_title(f'{method_name}\nBin Size: {bin_size}', fontsize=11, fontweight='bold')
            
            # Set x-axis to log scale if needed (thresholds span many orders of magnitude)
            if len(thresholds) > 1 and max(thresholds) / min(thresholds[thresholds > 0]) > 100:
                ax.set_xscale('log')
            
            # Add some styling
            ax.tick_params(axis='both', which='major', labelsize=9)
            
        else:
            # No data available
            ax.text(0.5, 0.5, f'No data for\n{method_name}\nBin {bin_size}', 
                   ha='center', va='center', transform=ax.transAxes, fontsize=10)
            ax.set_title(f'{method_name}\nBin Size: {bin_size}', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig("/home/dlipsey/MITLincolnLabs/Figures/Median_Error_vs_Threshold_Line_Plots")
plt.show()

# Print summary of what was plotted
print("=== LINE PLOTS SUMMARY ===")
for method_name, method_data in zip(methods, data_sources):
    print(f"\n{method_name}:")
    for bin_size in selected_bin_sizes:
        if bin_size in method_data:
            n_points = len(method_data[bin_size]['thresholds'])
            min_error = min(method_data[bin_size]['median_errors'])
            print(f"  Bin {bin_size}: {n_points} data points, min error: {min_error:.1f}%")
        else:
            print(f"  Bin {bin_size}: No data available")

In [ ]:
# Configuration - easily changeable threshold values
selected_thresholds = [0.05, 1, 5]  # Change these values as needed

# Extract data for the selected threshold values
def extract_data_for_thresholds(results_df, thresholds):
    """Extract median percent error data for selected threshold values"""
    data = {}
    for threshold in thresholds:
        threshold_data = results_df[results_df['Threshold'] == threshold].copy()
        if not threshold_data.empty:
            # Sort by bin size for proper line plotting
            threshold_data = threshold_data.sort_values('BinSize')
            data[threshold] = {
                'bin_sizes': threshold_data['BinSize'].values,
                'median_errors': threshold_data['Test_Median_Percent_Error'].values
            }
    return data

# Extract data for all three methods
spectra_data_by_thresh = extract_data_for_thresholds(df_percent_error_results, selected_thresholds)
chemnet_data_by_thresh = extract_data_for_thresholds(df_chemnet_percent_error_results, selected_thresholds)
cond_enc_data_by_thresh = extract_data_for_thresholds(df_cond_percent_error_results, selected_thresholds)

# Calculate global y-axis limits for standardized scale
all_errors = []
data_sources_for_limits = [spectra_data_by_thresh, chemnet_data_by_thresh, cond_enc_data_by_thresh]
for method_data in data_sources_for_limits:
    for threshold_data in method_data.values():
        if 'median_errors' in threshold_data:
            all_errors.extend(threshold_data['median_errors'])

if all_errors:  # Only if we have data
    y_min = min(all_errors)
    y_max = max(all_errors) 
    y_margin = (y_max - y_min) * 0.05  # 5% margin
    y_limits = (y_min - y_margin, y_max + y_margin)
else:
    y_limits = None

# Create 3x3 grid of line plots
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Median Absolute Percent Error vs Bin Size Values (Line Plots)', fontsize=16, fontweight='bold')

methods = ['Spectra', 'ChemNet', 'Conditional Encoder']
data_sources = [spectra_data_by_thresh, chemnet_data_by_thresh, cond_enc_data_by_thresh]
colors = ['blue', 'red', 'green']

for method_idx, (method_name, method_data, color) in enumerate(zip(methods, data_sources, colors)):
    for thresh_idx, threshold in enumerate(selected_thresholds):
        ax = axes[method_idx, thresh_idx]
        
        if threshold in method_data:
            bin_sizes = method_data[threshold]['bin_sizes']
            errors = method_data[threshold]['median_errors']
            
            # Plot line with markers
            ax.plot(bin_sizes, errors, marker='o', linewidth=2, markersize=6, 
                   color=color, alpha=0.7, label=f'Threshold {threshold}')
            
            # Add grid
            ax.grid(True, alpha=0.3)
            
            # Set labels
            if method_idx == 2:  # Bottom row
                ax.set_xlabel('Bin Size', fontsize=10)
            if thresh_idx == 0:  # Left column
                ax.set_ylabel('Median Absolute % Error', fontsize=10)
            
            # Set title
            ax.set_title(f'{method_name}\nThreshold: {threshold}', fontsize=11, fontweight='bold')
            
            # Set x-axis to log scale if needed (bin sizes span many orders of magnitude)
            if len(bin_sizes) > 1 and max(bin_sizes) / min(bin_sizes) > 100:
                ax.set_xscale('log')
            
            # Add some styling
            ax.tick_params(axis='both', which='major', labelsize=9)
            
            # Standardize y-axis scale
            if y_limits:
                ax.set_ylim(y_limits)
            
        else:
            # No data available
            ax.text(0.5, 0.5, f'No data for\n{method_name}\nThreshold {threshold}', 
                   ha='center', va='center', transform=ax.transAxes, fontsize=10)
            ax.set_title(f'{method_name}\nThreshold: {threshold}', fontsize=11, fontweight='bold')
            
            # Apply same y-limits to empty plots for consistency
            if y_limits:
                ax.set_ylim(y_limits)

plt.tight_layout()
plt.savefig("/home/dlipsey/MITLincolnLabs/Figures/Median_Error_vs_Bin_Size_Line_Plots")
plt.show()

# Print summary of what was plotted
print("=== LINE PLOTS SUMMARY ===")
for method_name, method_data in zip(methods, data_sources):
    print(f"\n{method_name}:")
    for threshold in selected_thresholds:
        if threshold in method_data:
            n_points = len(method_data[threshold]['bin_sizes'])
            min_error = min(method_data[threshold]['median_errors'])
            print(f"  Threshold {threshold}: {n_points} data points, min error: {min_error:.1f}%")
        else:
            print(f"  Threshold {threshold}: No data available")

In [ ]:
# Create 3x3 grid of bar plots
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Median Absolute Percent Error vs Threshold Values (Bar Plots)', fontsize=16, fontweight='bold')

methods = ['Spectra', 'ChemNet', 'Conditional Encoder']
data_sources = [spectra_data, chemnet_data, cond_enc_data]
colors = ['skyblue', 'lightcoral', 'lightgreen']

for method_idx, (method_name, method_data, color) in enumerate(zip(methods, data_sources, colors)):
    for bin_idx, bin_size in enumerate(selected_bin_sizes):
        ax = axes[method_idx, bin_idx]
        
        if bin_size in method_data:
            thresholds = method_data[bin_size]['thresholds']
            errors = method_data[bin_size]['median_errors']
            
            # Create bar plot
            bars = ax.bar(range(len(thresholds)), errors, color=color, 
                         alpha=0.7, edgecolor='black', linewidth=0.5)
            
            # Customize x-axis labels
            threshold_labels = [f'{t}' if t != 0 else '0' for t in thresholds]
            ax.set_xticks(range(len(thresholds)))
            ax.set_xticklabels(threshold_labels, rotation=45, ha='right', fontsize=8)
            
            # Add value labels on top of bars
            for i, (bar, error) in enumerate(zip(bars, errors)):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                       f'{error:.1f}', ha='center', va='bottom', fontsize=8)
            
            # Add grid
            ax.grid(True, alpha=0.3, axis='y')
            
            # Set labels
            if method_idx == 2:  # Bottom row
                ax.set_xlabel('Threshold Value', fontsize=10)
            if bin_idx == 0:  # Left column
                ax.set_ylabel('Median Absolute % Error', fontsize=10)
            
            # Set title
            ax.set_title(f'{method_name}\nBin Size: {bin_size}', fontsize=11, fontweight='bold')
            
            # Add some styling
            ax.tick_params(axis='both', which='major', labelsize=9)
            
            # Set y-axis limits with some headroom for text labels
            max_error = max(errors)
            ax.set_ylim(0, max_error * 1.15)
            
        else:
            # No data available
            ax.text(0.5, 0.5, f'No data for\n{method_name}\nBin {bin_size}', 
                   ha='center', va='center', transform=ax.transAxes, fontsize=10)
            ax.set_title(f'{method_name}\nBin Size: {bin_size}', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig("/home/dlipsey/MITLincolnLabs/Figures/Median_Error_vs_Threshold_Bar_Plots")
plt.show()

# Print summary of what was plotted
print("=== BAR PLOTS SUMMARY ===")
for method_name, method_data in zip(methods, data_sources):
    print(f"\n{method_name}:")
    for bin_size in selected_bin_sizes:
        if bin_size in method_data:
            n_points = len(method_data[bin_size]['thresholds'])
            min_error = min(method_data[bin_size]['median_errors'])
            max_error = max(method_data[bin_size]['median_errors'])
            print(f"  Bin {bin_size}: {n_points} bars, error range: {min_error:.1f}% - {max_error:.1f}%")
        else:
            print(f"  Bin {bin_size}: No data available")

# Show which bin sizes can be easily changed
print(f"\n=== CONFIGURATION ===")
print(f"Current selected bin sizes: {selected_bin_sizes}")
print("To change bin sizes, modify the 'selected_bin_sizes' variable at the top of the first code block")
print("Available bin sizes in data:")
all_available_bins = set()
if not df_percent_error_results.empty:
    all_available_bins.update(df_percent_error_results['BinSize'].unique())
if not df_chemnet_percent_error_results.empty:
    all_available_bins.update(df_chemnet_percent_error_results['BinSize'].unique())
if not df_cond_percent_error_results.empty:
    all_available_bins.update(df_cond_percent_error_results['BinSize'].unique())
print(f"Available: {sorted(list(all_available_bins))}")

In [ ]:
# Create 3x3 grid of bar plots
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Median Absolute Percent Error vs Bin Size Values (Bar Plots)', fontsize=16, fontweight='bold')

methods = ['Spectra', 'ChemNet', 'Conditional Encoder']
data_sources = [spectra_data_by_thresh, chemnet_data_by_thresh, cond_enc_data_by_thresh]
colors = ['skyblue', 'lightcoral', 'lightgreen']

for method_idx, (method_name, method_data, color) in enumerate(zip(methods, data_sources, colors)):
    for thresh_idx, threshold in enumerate(selected_thresholds):
        ax = axes[method_idx, thresh_idx]
        
        if threshold in method_data:
            bin_sizes = method_data[threshold]['bin_sizes']
            errors = method_data[threshold]['median_errors']
            
            # Create bar plot
            bars = ax.bar(range(len(bin_sizes)), errors, color=color, 
                         alpha=0.7, edgecolor='black', linewidth=0.5)
            
            # Customize x-axis labels
            bin_size_labels = [f'{bs}' for bs in bin_sizes]
            ax.set_xticks(range(len(bin_sizes)))
            ax.set_xticklabels(bin_size_labels, rotation=45, ha='right', fontsize=8)
            
            # Add value labels on top of bars
            for i, (bar, error) in enumerate(zip(bars, errors)):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                       f'{error:.1f}', ha='center', va='bottom', fontsize=8)
            
            # Add grid
            ax.grid(True, alpha=0.3, axis='y')
            
            # Set labels
            if method_idx == 2:  # Bottom row
                ax.set_xlabel('Bin Size', fontsize=10)
            if thresh_idx == 0:  # Left column
                ax.set_ylabel('Median Absolute % Error', fontsize=10)
            
            # Set title
            ax.set_title(f'{method_name}\nThreshold: {threshold}', fontsize=11, fontweight='bold')
            
            # Add some styling
            ax.tick_params(axis='both', which='major', labelsize=9)
            
            # Set y-axis limits with some headroom for text labels
            max_error = max(errors)
            ax.set_ylim(0, max_error * 1.15)
            
        else:
            # No data available
            ax.text(0.5, 0.5, f'No data for\n{method_name}\nThreshold {threshold}', 
                   ha='center', va='center', transform=ax.transAxes, fontsize=10)
            ax.set_title(f'{method_name}\nThreshold: {threshold}', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig("/home/dlipsey/MITLincolnLabs/Figures/Median_Error_vs_Bin_Size_Bar_Plots")
plt.show()

# Print summary of what was plotted
print("=== BAR PLOTS SUMMARY ===")
for method_name, method_data in zip(methods, data_sources):
    print(f"\n{method_name}:")
    for threshold in selected_thresholds:
        if threshold in method_data:
            n_points = len(method_data[threshold]['bin_sizes'])
            min_error = min(method_data[threshold]['median_errors'])
            max_error = max(method_data[threshold]['median_errors'])
            print(f"  Threshold {threshold}: {n_points} bars, error range: {min_error:.1f}% - {max_error:.1f}%")
        else:
            print(f"  Threshold {threshold}: No data available")

# Show which threshold values can be easily changed
print(f"\n=== CONFIGURATION ===")
print(f"Current selected thresholds: {selected_thresholds}")
print("To change thresholds, modify the 'selected_thresholds' variable at the top of the first code block")
print("Available threshold values in data:")
all_available_thresholds = set()
if not df_percent_error_results.empty:
    all_available_thresholds.update(df_percent_error_results['Threshold'].unique())
if not df_chemnet_percent_error_results.empty:
    all_available_thresholds.update(df_chemnet_percent_error_results['Threshold'].unique())
if not df_cond_percent_error_results.empty:
    all_available_thresholds.update(df_cond_percent_error_results['Threshold'].unique())
print(f"Available: {sorted(list(all_available_thresholds))}")

In [ ]:
# Bin size and threshold size for conditional encoder
bin_size = '0_05'        # Format like dataset name strings  
threshold = '0_1'        # Ditto
# =====================================================================

# Load the true ChemNet embeddings for SMILES matching
true_chemnet_path = '/home/dlipsey/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QQpos_no_repeats.csv'
true_chemnet_df = pd.read_csv(true_chemnet_path)

# Construct conditional encoder dataset name
cond_dataset_name = f'cond_enc_bin{bin_size}_thresh{threshold}_df3_QQpos_spectra'

# Load the conditional encoder dataset from the pickle file
cond_enc_folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/cond_enc_outputs"
cond_enc_file_path = os.path.join(cond_enc_folder, f"{cond_dataset_name}.pkl")
cond_enc_data = pd.read_pickle(cond_enc_file_path)

print(f"Loaded conditional encoder dataset: {cond_dataset_name}")
print(f"Shape: {cond_enc_data.shape}")
print(f"Columns: {cond_enc_data.columns.tolist()}")

# Get the conditional encoder embedding columns (exclude SMILES, Response, and other non-embedding columns)
numeric_cols = cond_enc_data.select_dtypes(include=[np.number]).columns.tolist()
# Look for conditional encoder embedding columns - they might be named differently
# Check what columns exist to identify the embedding columns
emb_cols = []
for col in cond_enc_data.columns:
    if 'emb_' in col or 'embed' in col.lower() or (col not in ['SMILES_spectra', 'Response', 'log_response', 'index_id', 'cond_tox_pred']):
        if pd.api.types.is_numeric_dtype(cond_enc_data[col]):
            emb_cols.append(col)

# If we can't find specific embedding columns, use all numeric columns except known non-embedding ones
if len(emb_cols) == 0:
    emb_cols = [col for col in numeric_cols if col not in ['Response', 'log_response', 'index_id', 'cond_tox_pred']]

# Get the true embedding columns to match dimensions
true_emb_cols = [col for col in true_chemnet_df.columns if col != 'SMILES']
n_true_features = len(true_emb_cols)

# Ensure we only use the same number of features as the true embeddings (if needed)
if len(emb_cols) > n_true_features:
    emb_cols_matched = emb_cols[:n_true_features]
else:
    emb_cols_matched = emb_cols

print(f"Conditional encoder embedding columns found: {len(emb_cols)}")
print(f"True embedding columns: {len(true_emb_cols)}")
print(f"Using {len(emb_cols_matched)} matched columns for PCA")

# Get the embedding data
X_cond_enc = cond_enc_data[emb_cols_matched].values

# Create chronic outlier labels using the previously identified chronic outlier IDs
if 'chronic_cond_outlier_ids' in globals():
    # Check if index_id exists in the conditional encoder data
    if 'index_id' in cond_enc_data.columns:
        cond_enc_ids = cond_enc_data['index_id'].tolist()
        is_chronic_outlier = [idx in chronic_cond_outlier_ids for idx in cond_enc_ids]
    else:
        # Use DataFrame index as fallback
        cond_enc_ids = cond_enc_data.index.tolist()
        is_chronic_outlier = [idx in chronic_cond_outlier_ids for idx in cond_enc_ids]
else:
    # If chronic outliers not defined, create dummy data
    print("Warning: chronic_cond_outlier_ids not found, using dummy data")
    is_chronic_outlier = [False] * len(cond_enc_data)

# Perform PCA
pca = PCA(n_components=2)
X_cond_enc_pca = pca.fit_transform(X_cond_enc)

# Find true embeddings by matching SMILES
true_embeddings_pca = []
true_smiles_found = []

for idx, row in cond_enc_data.iterrows():
    smiles = row['SMILES_spectra']
    # Find matching SMILES in true ChemNet data
    true_match = true_chemnet_df[true_chemnet_df['SMILES'] == smiles]
    
    if not true_match.empty:
        # Get the true embedding (exclude SMILES column)
        true_emb_cols = [col for col in true_chemnet_df.columns if col != 'SMILES']
        true_embedding = true_match[true_emb_cols].values[0]
        
        # Ensure dimension matching
        if len(true_embedding) > len(emb_cols_matched):
            true_embedding = true_embedding[:len(emb_cols_matched)]
        
        # Transform the true embedding using the same PCA
        true_embedding_pca = pca.transform(true_embedding.reshape(1, -1))[0]
        true_embeddings_pca.append(true_embedding_pca)
        true_smiles_found.append(smiles)

true_embeddings_pca = np.array(true_embeddings_pca)

In [ ]:
# PLOT 1: Colored by Chronic Outlier Status
plt.figure(figsize=(12, 8))

# Plot non-chronic outliers in blue
non_chronic_mask = ~np.array(is_chronic_outlier)
plt.scatter(X_cond_enc_pca[non_chronic_mask, 0], 
           X_cond_enc_pca[non_chronic_mask, 1],
           c='blue', marker='o', alpha=0.7, s=50, label='Non-chronic outliers')

# Plot chronic outliers in red
chronic_mask = np.array(is_chronic_outlier)
plt.scatter(X_cond_enc_pca[chronic_mask, 0], 
           X_cond_enc_pca[chronic_mask, 1],
           c='red', marker='o', alpha=0.7, s=50, label='Chronic outliers')

# Plot true embeddings as black squares
if len(true_embeddings_pca) > 0:
    plt.scatter(true_embeddings_pca[:, 0], true_embeddings_pca[:, 1], 
               c='black', marker='s', s=25, label=f'True embeddings ({len(true_embeddings_pca)})', 
               linewidth=2)

# Customize the plot
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title(f'2D PCA of Conditional Encoder Embeddings: {cond_dataset_name}\nColored by Chronic Outlier Status')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(f"/home/dlipsey/MITLincolnLabs/Figures/cond_enc_bin{bin_size}_thresh{threshold}_chronic_outlier_PCA.png")
plt.show()

In [ ]:
# PLOT 2: Colored by SMILES with True vs Predicted Legend
plt.figure(figsize=(12, 8))

# Get unique SMILES for color mapping
unique_smiles = cond_enc_data['SMILES_spectra'].unique()
n_smiles = len(unique_smiles)

# Create color map for SMILES
import matplotlib.cm as cm
colors = cm.tab20(np.linspace(0, 1, min(n_smiles, 50)))
if n_smiles > 20:
    # If more than 20 SMILES, cycle through colors
    colors = cm.tab20(np.linspace(0, 1, 20))
    color_map = {smiles: colors[i % 20] for i, smiles in enumerate(unique_smiles)}
else:
    color_map = {smiles: colors[i] for i, smiles in enumerate(unique_smiles)}

# Plot predicted embeddings colored by SMILES
predicted_colors = [color_map[smiles] for smiles in cond_enc_data['SMILES_spectra']]
plt.scatter(X_cond_enc_pca[:, 0], X_cond_enc_pca[:, 1],
           c=predicted_colors, marker='o', alpha=0.7, s=50)

# Plot true embeddings as black squares
if len(true_embeddings_pca) > 0:
    plt.scatter(true_embeddings_pca[:, 0], true_embeddings_pca[:, 1], 
               c='black', marker='s', s=25, linewidth=2)

# Create simplified legend for true vs predicted
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='lightgray', edgecolor='black', label='Predicted Embeddings'),
    Patch(facecolor='black', edgecolor='black', label=f'True Embeddings ({len(true_embeddings_pca)})')
]

# Customize the plot
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title(f'2D PCA of Conditional Encoder Embeddings: {cond_dataset_name}\nColored by SMILES (True vs Predicted)')
plt.legend(handles=legend_elements, loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Print additional statistics
print(f"\nSMILES Analysis:")
print(f"Unique SMILES: {n_smiles}")
print(f"Average samples per SMILES: {len(cond_enc_data) / n_smiles:.1f}")

plt.savefig(f"/home/dlipsey/MITLincolnLabs/Figures/cond_enc_bin{bin_size}_thresh{threshold}_SMILES_PCA.png")
plt.show()